# Coefficient-Decoupled MPOs as an Interface to LCU Circuits — numerical pipeline

Companion notebook for **arXiv:2603.24822** (submitted to *Quantum Science and Technology*, QST-105164).

Pipeline (matches the paper's workflow):

1. **Molecular system** — PySCF/OpenFermion active-space Hamiltonian (paper example: C$_2$H$_2$/STO-3G, 20 qubits).
2. **Coefficient-decoupled MPO** — left/right symbolic Pauli-fragment graphs + coefficient bridge across the central cut (paper Sec. III, Fig. 2).
3. **DMRG reference state** — pretrained MPS at bond dimension $\chi \in \{8, 32\}$.
4. **Perfect sampling of Pauli strings from the pretrained MPS** — distribution $\Pi(P) = \langle\psi|P|\psi\rangle^2 / 2^N$ (paper Sec. VI / Appendix F, Algorithm 2).
5. **Pool curation** — keep all strings containing $X$/$Y$, plus the top-$k$ purely diagonal strings.
6. **Bridge-coefficient optimization** — adaptive generalized eigenproblem on the bridge (Option C, LOBPCG/dense ED).
7. **Paper figures** — Fig. 4 (polar Pauli maps) and Fig. 5 (energy vs $N_\mathrm{samples}$), with raw data stored in `results/`.

> **Naming note:** the submitted paper labels the molecule "C$_2$H$_4$", but the geometry actually used
> (`C2H2_sto3g_frozen`) is **acetylene C$_2$H$_2$** — to be corrected in the revision. A genuine
> `C2H4_sto3g_frozen` entry (ethylene) is provided below as a second system for the revision.


## 0. Setup and reproducibility

All hyperparameters and random seeds live in the configuration cell below (referee request:
explicit seeds, DMRG settings, and software stack).

The notebook runs on **CPU or GPU**: set `DEVICE` in the configuration cell
(`"auto"` picks the GPU automatically when CuPy is available, e.g. on a Colab GPU runtime).
DMRG/MPS kernels run on the selected device; the graph construction and bridge optimization
run on CPU.


In [ ]:
# ----- Device / backend selection -----
# Must be set BEFORE importing qu_alg_qu_sim_tensor_networks (read at import time).
# "cpu"  -> NumPy everywhere (recommended: the bridge optimizer is CPU-bound anyway)
# "gpu"  -> CuPy for DMRG/MPS kernels (requires CUDA + cupy, e.g. a Colab GPU runtime)
# "auto" -> CuPy if it imports (GPU runtime), else NumPy. JAX is never used here.
import os
DEVICE = "auto"
os.environ["DMRG_BACKEND"] = {"cpu": "numpy", "gpu": "cupy", "auto": "auto"}[DEVICE]

# ----- Global configuration: every seed and hyperparameter in one place -----
MOLECULE_KEY   = "C2H2_sto3g_frozen"   # paper system (acetylene, 20 qubits).
                                       # For the revision, also run "C2H4_sto3g_frozen".
TRANSFORM      = "jw"                  # fermion-to-qubit mapping
TOP_K_TERMS    = 3000                  # Pauli terms kept for the symbolic graph
MIN_ABS_COEFF  = 1e-12

# Hamiltonian MPO construction
MPO_EPS        = 1e-7                  # truncation threshold
MPO_CHI_MAX    = 200

# DMRG reference state (paper uses chi = 8 and 32)
CHI_DMRG       = 8
DMRG_SWEEPS    = 15
DMRG_EPS       = 1e-11
DMRG_INIT_CHI  = 4
DMRG_SEED      = 1234

# Pauli-string sampling from the pretrained MPS (paper Algorithm 2)
N_PAULI_SAMPLES = 1000
SAMPLING_SEED   = 0

# Pool curation rule: keep ALL strings containing X/Y + top-k diagonal (I/Z-only) strings
TOP_K_DIAGONAL  = 5

# Bridge optimization (Option C)
BRIDGE_SEED     = 2


In [ ]:
# ----- Google Colab bootstrap (no-op on a local machine) -----
# Workflow: upload the repository to Google Drive, run this notebook on a Colab
# GPU runtime (T4/A100). Outputs (figures + data) are written to Drive and can
# also be downloaded as a single zip at the end of the notebook.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

USE_DRIVE  = True   # mount Google Drive: install the repo from Drive, save outputs to Drive
DRIVE_REPO = "/content/drive/MyDrive/qu_alg_qu_sim_tensor_networks"  # repo copy on your Drive
DRIVE_OUT  = "/content/drive/MyDrive/mpo_lcu_outputs"                # figures + data land here

# Output locations (overridden below when Drive is mounted)
RESULTS_DIR = "results"
FIGS_DIR    = "."

if IN_COLAB:
    import importlib.util, shutil, subprocess

    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        RESULTS_DIR = f"{DRIVE_OUT}/results"
        FIGS_DIR    = f"{DRIVE_OUT}/figs"

    if importlib.util.find_spec("qu_alg_qu_sim_tensor_networks") is None:
        # Pin numpy/pandas to the PREINSTALLED Colab versions inside the same pip
        # command -- otherwise the resolver upgrades them and breaks every compiled
        # package (sklearn, cupy, ...) until the runtime is restarted.
        import numpy as _np
        _pins = f"numpy=={_np.__version__}"
        try:
            import pandas as _pd
            _pins += f" pandas=={_pd.__version__}"
        except Exception:
            pass
        !pip -q install {_pins} pyscf openfermion openfermionpyscf qutip qutip-qip
        if USE_DRIVE:
            !pip -q install --no-deps {DRIVE_REPO}
        else:
            !pip -q install --no-deps git+https://github.com/y-javanmard/qu_alg_qu_sim_tensor_networks.git
        importlib.invalidate_caches()

    # On a GPU runtime (T4/A100), ensure CuPy so DEVICE="auto"/"gpu" can use it.
    has_gpu = shutil.which("nvidia-smi") is not None and \
        subprocess.run(["nvidia-smi", "-L"], capture_output=True).returncode == 0
    if has_gpu and importlib.util.find_spec("cupy") is None:
        !pip -q install cupy-cuda12x
    print("Colab GPU runtime:", has_gpu)

os.makedirs(RESULTS_DIR, exist_ok=True)
if FIGS_DIR != ".":
    os.makedirs(FIGS_DIR, exist_ok=True)
print("Running in Colab:", IN_COLAB)
print("RESULTS_DIR:", RESULTS_DIR)
print("FIGS_DIR   :", FIGS_DIR)

# ----- Finalize compute backend (runs on Colab AND locally) -----
# Never select JAX: the DMRG/MPS kernels use in-place array assignment, which JAX
# arrays forbid. Resolve "auto" ourselves to CuPy (if it truly imports) else NumPy.
if DEVICE == "auto":
    try:
        import cupy as _cp  # noqa: F401
        os.environ["DMRG_BACKEND"] = "cupy"
    except Exception:
        os.environ["DMRG_BACKEND"] = "numpy"
print("DMRG_BACKEND:", os.environ["DMRG_BACKEND"])


In [ ]:
# ----- Software stack (record versions for the reproducibility package) -----
import sys, platform, importlib

print("Python:", sys.version.replace("\n", " "))
print("Platform:", platform.platform())
for pkg in ["numpy", "scipy", "matplotlib", "networkx", "sklearn",
            "openfermion", "openfermionpyscf", "pyscf"]:
    try:
        m = importlib.import_module(pkg)
        print(f"{pkg:18s} {getattr(m, '__version__', '?')}")
    except Exception as exc:
        print(f"{pkg:18s} NOT AVAILABLE ({exc})")

from qu_alg_qu_sim_tensor_networks import backend as qbackend
print(f"compute backend    : {qbackend.BACK}  (DEVICE={DEVICE!r}, DMRG_BACKEND={os.environ.get('DMRG_BACKEND')})")

# Fail fast on a stale/incompatible backend instead of crashing deep in DMRG.
# JAX arrays are immutable and break the in-place MPS/DMRG kernels; and if the
# backend module was imported before DEVICE took effect, it will not match.
_want = {"cpu": "numpy", "gpu": "cupy", "auto": None}[DEVICE]
if qbackend.BACK == "jax" or (_want is not None and qbackend.BACK != _want):
    raise RuntimeError(
        f"Selected backend is '{qbackend.BACK}', but DEVICE={DEVICE!r} expects "
        f"'{_want or 'cupy/numpy'}'. The backend was imported before the setting "
        "took effect (or resolved to JAX, which is unsupported here).\n\n"
        ">>> FIX: Runtime -> Restart runtime, then Run all. <<<\n"
        "(Re-running cells is not enough: the backend is read only once at import.)"
    )


In [ ]:
import copy
import json
import os
import time

import numpy as np
import matplotlib.pyplot as plt

import qu_alg_qu_sim_tensor_networks.hamiltonians as ham
from qu_alg_qu_sim_tensor_networks.hamiltonians import Hamiltonian, get_mpo
from qu_alg_qu_sim_tensor_networks import graph_builder_new_version as gb
from qu_alg_qu_sim_tensor_networks import mpo_builder as qmpo
from qu_alg_qu_sim_tensor_networks import mps_helper, mps, dmrg
from qu_alg_qu_sim_tensor_networks.mps import MPS


In [ ]:
# ----- small helpers (random Pauli-sum generator + MPO bond-dimension reader) -----
_PAULIS = np.array(list("IXYZ"), dtype="<U1")


def add_string_term(H: Hamiltonian, n: int, sites, ops, coeff: complex):
    """Add a term defined by ops on given sites, identity elsewhere."""
    s = ["I"] * n
    for i, ch in zip(sites, ops):
        s[int(i)] = str(ch)
    s = "".join(s)
    if all(ch == "I" for ch in s):
        return
    H[s] = H[s] + complex(coeff)


def build_random_like_renormalizer(n: int, n_terms: int, rng: np.random.Generator):
    """
    Match the renormalizer-style generation:
      - k uniform in 1..n
      - sites sampled w/o replacement
      - ops sampled with replacement from {I,X,Y,Z}
      - coefficient ~ N(0,1) stored as complex
    """
    H = Hamiltonian(n)
    for _ in range(n_terms):
        k = int(rng.integers(1, n + 1))
        sites = rng.choice(n, size=k, replace=False)
        ops = rng.choice(_PAULIS, size=k, replace=True)
        coeff = float(rng.normal()) + 0j
        add_string_term(H, n, sites, ops, coeff)
    return H


def bonds_my(wmpo):
    if hasattr(wmpo, "bonds") and wmpo.bonds is not None:
        return list(map(int, wmpo.bonds))
    if hasattr(wmpo, "Ms") and wmpo.Ms is not None:
        return [1] + [int(wmpo.Ms[i].shape[-1]) for i in range(len(wmpo.Ms) - 1)] + [1]
    raise RuntimeError("Can't extract bonds from your MPO object.")


## 1. Molecular system

`build_case` runs PySCF through OpenFermion, applies the frozen-core active space, and maps to
qubits (Jordan–Wigner by default). The paper example is `C2H2_sto3g_frozen`
(frozen carbon 1s cores, 10 active orbitals → 20 qubits); `C2H4_sto3g_frozen` (ethylene) is
included for the revision's second system.


In [ ]:
#!/usr/bin/env python3
"""
molecule_cases.py

- In-memory PySCF (no save/load)
- Frozen-core + active above it:
    occupied_indices = range(frozen_core)
    active_indices   = range(frozen_core, frozen_core + n_active)
- Qubit mapping: jw / bk / bk_tree / bk_fast (auto-detected if installed)
- Exact ground state ONLY if n_qubits <= ED_MAX_QUBITS
- Graph-friendly Pauli dict: keep top-k terms by |coeff| (e.g. 200–400)
"""

import numpy as np
import scipy.linalg

from openfermion.chem import MolecularData
from openfermion.transforms import get_fermion_operator
from openfermion.linalg import get_ground_state, get_sparse_operator
from openfermionpyscf import run_pyscf
import openfermion as of

ED_MAX_QUBITS = 12


# -----------------------------
# Molecule library (kept + add H3/H4 chains + extra 6..20 qubit ladder)
# -----------------------------
MOLECULES = {
    "H2_sto3g": {
        "kind": "diatomic", "atoms": ("H", "H"), "r_default": 0.7414,
        "basis": "sto-3g", "charge": 0, "multiplicity": 1,
        "frozen_core": 0, "n_active": 2,   # 4 qubits
    },

    "BeH_sto3g_small": {
        "kind": "diatomic", "atoms": ("Be", "H"), "r_default": 1.34,
        "basis": "sto-3g", "charge": 0, "multiplicity": 2,
        "frozen_core": 1, "n_active": 3,   # 6 qubits
    },

    "H7_sto3g_small": {
        "kind": "linear_chain", "r_default": 0.9584, "angle_deg_default": 104.45,
        "basis": "sto-3g", "charge": 0, "multiplicity": 1,
        "frozen_core": 0, "n_active": 7,   # 14 qubits (comment in original said 12, but n_active=7 => 14)
    },

    "H8_chain_sto3g_big": {
        "kind": "linear_chain", "atoms": ("H",), "n_atoms": 8, "r_default": 2.0,
        "basis": "sto-3g", "charge": 0, "multiplicity": 1,
        "frozen_core": 0, "n_active": 8,   # 16 qubits
    },

    "LiH_sto3g_small": {
        "kind": "diatomic", "atoms": ("Li", "H"), "r_default": 1.0,
        "basis": "sto-3g", "charge": 0, "multiplicity": 1,
        "frozen_core": 1, "n_active": 2,   # 4 qubits
    },
    "LiH_sto3g_medium": {
        "kind": "diatomic", "atoms": ("Li", "H"), "r_default": 1.45,
        "basis": "sto-3g", "charge": 0, "multiplicity": 1,
        "frozen_core": 1, "n_active": 3,   # 6 qubits
    },

    "LiH_sto3g_full": {
        "kind": "diatomic", "atoms": ("Li", "H"), "r_default": 1.5,
        "basis": "sto-3g", "charge": 0, "multiplicity": 1,
        "frozen_core": 0, "n_active": 6,   # 12 qubits
    },
    "HF_sto3g_full": {
        "kind": "diatomic", "atoms": ("H", "F"), "r_default": 0.917,
        "basis": "sto-3g", "charge": 0, "multiplicity": 1,
        "frozen_core": 0, "n_active": 6,   # 12 qubits
    },

    # ---------------- NEW: H3 / H4 linear hydrogen chains ----------------
    # H3 has 3 electrons -> typically a doublet (multiplicity=2)
    "H3_chain_sto3g": {
        "kind": "linear_chain", "atoms": ("H",), "n_atoms": 3, "r_default": 1.2,
        "basis": "sto-3g", "charge": 0, "multiplicity": 2,
        "frozen_core": 0, "n_active": 3,   # 6 qubits
    },
    # H4 has 4 electrons -> typically a singlet (multiplicity=1)
    "H4_chain_sto3g": {
        "kind": "linear_chain", "atoms": ("H",), "n_atoms": 4, "r_default": 1.2,
        "basis": "sto-3g", "charge": 0, "multiplicity": 1,
        "frozen_core": 0, "n_active": 4,   # 8 qubits
    },

    "C2H2_sto3g_frozen": {
        "kind": "molecule",
        "atoms": ["C", "C", "H", "H"],
        "geometry": [
            ("C", (0.0, 0.0, 0.6)),
            ("C", (0.0, 0.0, -0.6)),
            ("H", (0.0, 0.0, 1.66)),
            ("H", (0.0, 0.0, -1.66))
        ],
        "basis": "sto-3g", "charge": 0, "multiplicity": 1,
        "frozen_core": 1,   # Freezes the 1s orbitals of Carbon
        "n_active": 10,     # 20 qubits
    },

    "C2H4_sto3g_frozen": {
        "kind": "molecule",
        "atoms": ["C", "C", "H", "H", "H", "H"],
        "geometry": [
            ("C", (0.0, 0.0, 0.6695)),
            ("C", (0.0, 0.0, -0.6695)),
            ("H", (0.0, 0.9289, 1.2321)),
            ("H", (0.0, -0.9289, 1.2321)),
            ("H", (0.0, 0.9289, -1.2321)),
            ("H", (0.0, -0.9289, -1.2321)),
        ],
        "basis": "sto-3g", "charge": 0, "multiplicity": 1,
        "frozen_core": 2,   # freeze both carbon 1s orbitals
        "n_active": 10,     # 20 qubits: same register size as the C2H2 case
    },

    "N2_sto3g_full": {
        "kind": "diatomic",
        "atoms": ("N", "N"),
        "r_default": 1.098,
        "basis": "sto-3g", "charge": 0, "multiplicity": 1,
        "frozen_core": 0,
        "n_active": 10,     # 20 qubits
    },

    "H10_sto3g": {
        "kind": "molecule",
        "atoms": ["H"] * 10,
        "geometry": [("H", (0.0, 0.0, i * 1.0)) for i in range(10)],
        "basis": "sto-3g", "charge": 0, "multiplicity": 1,
        "frozen_core": 0,
        "n_active": 10,     # 20 qubits
    },

    # ============================================================
    # ADDITIONAL "DMRG-friendly" ladder (6 -> 20 qubits)
    #   NOTE: For >12 qubits you won’t ED; use min_abs_coeff + top_k_terms.
    #   Hydrogen chains at larger spacing tend to have many small coefficients,
    #   which makes it easy to prune to <1500 kept terms for your graph/bridge.
    # ============================================================

    # 10 qubits (closed-shell via cation)
    "H5_chain_sto3g_cation": {
        "kind": "linear_chain", "atoms": ("H",), "n_atoms": 5, "r_default": 1.6,
        "basis": "sto-3g", "charge": +1, "multiplicity": 1,
        "frozen_core": 0, "n_active": 5,   # 10 qubits
    },

    # 12 qubits (ED boundary)
    "H6_chain_sto3g": {
        "kind": "linear_chain", "atoms": ("H",), "n_atoms": 6, "r_default": 1.6,
        "basis": "sto-3g", "charge": 0, "multiplicity": 1,
        "frozen_core": 0, "n_active": 6,   # 12 qubits
    },

    # 14 qubits (sparse-ish, closed-shell via cation)
    "H7_chain_sto3g_cation_sparse": {
        "kind": "linear_chain", "atoms": ("H",), "n_atoms": 7, "r_default": 2.0,
        "basis": "sto-3g", "charge": +1, "multiplicity": 1,
        "frozen_core": 0, "n_active": 7,   # 14 qubits
    },

    # 14 qubits (neutral open-shell)
    "H7_chain_sto3g_neutral": {
        "kind": "linear_chain", "atoms": ("H",), "n_atoms": 7, "r_default": 2.0,
        "basis": "sto-3g", "charge": 0, "multiplicity": 2,
        "frozen_core": 0, "n_active": 7,   # 14 qubits
    },

    # 16 qubits (dense-ish benchmark)
    "H8_chain_sto3g_dense": {
        "kind": "linear_chain", "atoms": ("H",), "n_atoms": 8, "r_default": 1.0,
        "basis": "sto-3g", "charge": 0, "multiplicity": 1,
        "frozen_core": 0, "n_active": 8,   # 16 qubits
    },

    # 18 qubits (sparse-ish, closed-shell via cation)
    "H9_chain_sto3g_cation_sparse": {
        "kind": "linear_chain", "atoms": ("H",), "n_atoms": 9, "r_default": 2.0,
        "basis": "sto-3g", "charge": +1, "multiplicity": 1,
        "frozen_core": 0, "n_active": 9,   # 18 qubits
    },

    # 18 qubits (neutral open-shell)
    "H9_chain_sto3g_neutral": {
        "kind": "linear_chain", "atoms": ("H",), "n_atoms": 9, "r_default": 2.0,
        "basis": "sto-3g", "charge": 0, "multiplicity": 2,
        "frozen_core": 0, "n_active": 9,   # 18 qubits
    },

    # 14 qubits (a non-chain example; STO-3G water often has 7 spatial orbitals -> 14 qubits)
    "H2O_sto3g_full_active": {
        "kind": "water", "r_default": 0.9584, "angle_deg_default": 104.45,
        "basis": "sto-3g", "charge": 0, "multiplicity": 1,
        "frozen_core": 0, "n_active": 7,   # typically 14 qubits
    },
}


# -----------------------------
# Geometry builders
# -----------------------------
def build_geometry(spec: dict, r: float):
    kind = spec["kind"]
    r = float(r)

    if kind == "diatomic":
        A, B = spec["atoms"]
        return [(A, (0.0, 0.0, 0.0)), (B, (0.0, 0.0, r))]

    if kind == "linear_chain":
        A = spec.get("atoms", ("H",))[0]
        # robust fallback: if n_atoms isn't provided, use n_active
        n_atoms = spec.get("n_atoms", None)
        if n_atoms is None:
            n_atoms = spec.get("n_active", None)
        if n_atoms is None:
            raise ValueError("linear_chain requires spec['n_atoms'] or spec['n_active']")
        n_atoms = int(n_atoms)

        z0 = -0.5 * (n_atoms - 1) * r
        return [(A, (0.0, 0.0, z0 + i * r)) for i in range(n_atoms)]

    if kind == "water":
        angle_deg = float(spec.get("angle_deg_default", 104.45))
        theta = np.deg2rad(angle_deg)
        x = r * np.sin(theta / 2.0)
        z = r * np.cos(theta / 2.0)
        return [
            ("O", (0.0, 0.0, 0.0)),
            ("H", (+x, 0.0, z)),
            ("H", (-x, 0.0, z)),
        ]

    if kind == "molecule":
        # Preferred: explicit geometry list is provided.
        if "geometry" in spec:
            return spec["geometry"]

        # Optional fallback: line-place atoms with spacing r
        atoms = spec.get("atoms", None)
        if atoms is None:
            raise ValueError("kind='molecule' requires spec['geometry'] or spec['atoms'].")

        n_atoms = len(atoms)
        z0 = -0.5 * (n_atoms - 1) * r
        return [(atoms[i], (0.0, 0.0, z0 + i * r)) for i in range(n_atoms)]

    raise ValueError(f"Unknown kind: {kind}")


# -----------------------------
# OpenFermion transforms
# -----------------------------
def available_qubit_transforms():
    T = of.transforms
    candidates = {
        "jw": "jordan_wigner",
        "bk": "bravyi_kitaev",
        "bk_tree": "bravyi_kitaev_tree",
        "bk_fast": "bravyi_kitaev_fast",
    }
    out = {}
    for short, name in candidates.items():
        fn = getattr(T, name, None)
        if callable(fn):
            out[short] = fn
    return out


TRANSFORMS = available_qubit_transforms()


def fermion_to_qubit(fermion_h, n_qubits: int, transform: str):
    if transform not in TRANSFORMS:
        raise ValueError(f"Unknown transform={transform!r}. Available: {sorted(TRANSFORMS)}")
    fn = TRANSFORMS[transform]
    try:
        qh = fn(fermion_h, n_qubits)
    except TypeError:
        qh = fn(fermion_h)
    qh.compress()
    return qh


# -----------------------------
# Active space selection
# -----------------------------
def resolve_active_space_indices(mol: MolecularData, spec: dict):
    n_orb = int(mol.n_orbitals)
    frozen_core = int(spec["frozen_core"])
    n_active = int(spec["n_active"])

    if frozen_core < 0 or n_active <= 0:
        raise ValueError(f"Bad frozen_core/n_active: frozen_core={frozen_core}, n_active={n_active}")

    start = frozen_core
    stop = frozen_core + n_active

    if stop > n_orb:
        raise ValueError(
            f"Active space out of bounds: frozen_core+n_active={stop} > mol.n_orbitals={n_orb}. "
            f"(frozen_core={frozen_core}, n_active={n_active})"
        )

    occupied_indices = range(start)
    active_indices = range(start, stop)
    n_qubits = 2 * (stop - start)
    return occupied_indices, active_indices, n_qubits


def active_space_qubit_hamiltonian(mol: MolecularData, spec: dict, transform: str):
    occ, act, _ = resolve_active_space_indices(mol, spec)
    mh = mol.get_molecular_hamiltonian(occupied_indices=occ, active_indices=act)
    fermion_h = get_fermion_operator(mh)
    qubit_h = fermion_to_qubit(fermion_h, n_qubits=mh.n_qubits, transform=transform)
    return qubit_h, mh, list(occ), list(act)


# -----------------------------
# Exact GS only if small
# -----------------------------
def exact_ground_state_if_small(qubit_h, n_qubits: int):
    if n_qubits > ED_MAX_QUBITS:
        return None, None
    E0, psi0 = get_ground_state(get_sparse_operator(qubit_h))
    return float(E0), psi0


# -----------------------------
# Optional orbital rotation
# -----------------------------
def random_orbital_rotation(molecular_hamiltonian, seed: int = 1):
    n_orb = molecular_hamiltonian.n_qubits // 2
    n_var = n_orb * (n_orb - 1) // 2
    rng = np.random.default_rng(seed)
    angles = np.pi * (1.0 - 2.0 * rng.random(n_var))

    kappa = np.zeros((n_orb, n_orb), dtype=float)
    idx = 0
    for p in range(n_orb):
        for q in range(p + 1, n_orb):
            kappa[p, q] = angles[idx]
            kappa[q, p] = -angles[idx]
            idx += 1

    U = scipy.linalg.expm(kappa)
    molecular_hamiltonian.rotate_basis(U)
    return U


# -----------------------------
# Graph-friendly top-k Pauli dict
# -----------------------------
def qubitop_topk_pauli_dict(qubit_h, n_qubits: int, *, k: int = 300, min_abs: float = 0.0):
    items = []
    for ops, coeff in qubit_h.terms.items():
        coeff = complex(coeff)
        if abs(coeff) <= min_abs:
            continue
        s = ["I"] * n_qubits
        for (i, p) in ops:
            s[int(i)] = str(p)
        items.append(("".join(s), coeff))

    items.sort(key=lambda kv: abs(kv[1]), reverse=True)
    return dict(items[: int(k)])


# -----------------------------
# Main build_case (INTERFACE UNCHANGED)
# -----------------------------
def build_case(
    key: str,
    *,
    r: float | None = None,
    transform: str = "jw",
    rotate: bool = False,
    rotation_seed: int = 1,
    top_k_terms: int = 300,
    min_abs_coeff: float = 0.0,
):
    spec = MOLECULES[key]
        # r_default is optional (e.g. kind="molecule" with explicit geometry)
    rr = float(spec.get("r_default", 1.0)) if r is None else float(r)


    mol = MolecularData(
        geometry=build_geometry(spec, rr),
        basis=spec["basis"],
        multiplicity=spec["multiplicity"],
        charge=spec["charge"],
        description=f"r={rr}",
    )
    mol = run_pyscf(mol, run_scf=True, run_fci=False)

    qubit_h, mh, occ, act = active_space_qubit_hamiltonian(mol, spec, transform=transform)
    n_qubits = mh.n_qubits

    E0, psi0 = exact_ground_state_if_small(qubit_h, n_qubits)

    out = {
        "key": key,
        "transform": transform,
        "spec": spec,
        "r": rr,
        "mol": mol,
        "mh": mh,
        "occupied_indices": occ,
        "active_indices": act,
        "qubit_h": qubit_h,
        "n_qubits": n_qubits,
        "E0": E0,
        "psi0": psi0,
        "pauli_dict_topk": qubitop_topk_pauli_dict(
            qubit_h, n_qubits=n_qubits, k=top_k_terms, min_abs=min_abs_coeff
        ),
    }

    if rotate:
        U = random_orbital_rotation(mh, seed=rotation_seed)
        fermion_h_rot = get_fermion_operator(mh)
        qubit_h_rot = fermion_to_qubit(fermion_h_rot, n_qubits=mh.n_qubits, transform=transform)
        E0r, psi0r = exact_ground_state_if_small(qubit_h_rot, n_qubits)
        out.update({
            "rotation_matrix": U,
            "qubit_h_rot": qubit_h_rot,
            "E0_rot": E0r,
            "psi0_rot": psi0r,
            "pauli_dict_topk_rot": qubitop_topk_pauli_dict(
                qubit_h_rot, n_qubits=n_qubits, k=top_k_terms, min_abs=min_abs_coeff
            ),
        })

    return out


# -----------------------------
# Optional: quick scan helper (no interface change)
# -----------------------------
def quick_scan(keys, *, transform="jw", top_k_terms=1500, min_abs_coeff=1e-8):
    for k in keys:
        out = build_case(k, transform=transform, rotate=False, top_k_terms=top_k_terms, min_abs_coeff=min_abs_coeff)
        raw_terms = len(out["qubit_h"].terms)
        kept_terms = len(out["pauli_dict_topk"])
        print(f"{k:28s}  n_qubits={out['n_qubits']:2d}  raw_terms={raw_terms:7d}  kept={kept_terms:5d}  E0={out['E0']}")


In [ ]:
out = build_case(MOLECULE_KEY, transform=TRANSFORM, rotate=False,
                 top_k_terms=TOP_K_TERMS, min_abs_coeff=MIN_ABS_COEFF)

print(f"[{out['key']}] n_qubits={out['n_qubits']}  E0={out['E0']}")
print("HF energy       :", out["mol"].hf_energy)
print("kept graph terms:", len(out["pauli_dict_topk"]),
      "| raw qubit terms:", len(out["qubit_h"].terms))


### Hartree–Fock reference product state

In [ ]:
def hf_bits_msb_left_ones_then_zeros_from_out(out):
    """
    Returns HF bitstring as MSB-left (display order): '111...000...'
    consistent with your active-space JW ordering (2p=alpha, 2p+1=beta).

    For closed-shell, this becomes exactly 111..000 with
      n_occ_spin = 2*(n_e/2 - n_core)
    ones then zeros.
    """
    mol = out["mol"]
    n_qubits = int(out["n_qubits"])
    n_active = n_qubits // 2

    n_e = int(mol.n_electrons)
    mult = int(getattr(mol, "multiplicity", 1))
    spin2 = mult - 1

    n_alpha = (n_e + spin2) // 2
    n_beta  = (n_e - spin2) // 2

    n_core = len(out["occupied_indices"])  # frozen spatial orbitals
    n_alpha_act = n_alpha - n_core
    n_beta_act  = n_beta  - n_core

    if not (0 <= n_alpha_act <= n_active and 0 <= n_beta_act <= n_active):
        raise ValueError("Active electron counts inconsistent with active size.")

    n_occ_spin = n_alpha_act + n_beta_act  # number of 1s in HF in the active JW register

    # In MSB-left display order, the occupied HF spin-orbitals are the highest-index qubits
    # so it prints as 111..000 (like your example).
    bits = "1" * n_occ_spin + "0" * (n_qubits - n_occ_spin)
    return bits

hf_bits = hf_bits_msb_left_ones_then_zeros_from_out(out)
print("HF bits:", hf_bits)


In [ ]:
import numpy as np
from qu_alg_qu_sim_tensor_networks import mps as mps

def product_mps_Bs_from_bitstring(bitstring_msb_left):
    Bs = []
    Ss = []
    for b in bitstring_msb_left:
        B = np.zeros((1, 2, 1), dtype=np.complex128)  # (a_l, d, a_r)
        B[0, 1 if b == "1" else 0, 0] = 1.0
        Bs.append(B)
        Ss.append(np.array([1.0], dtype=float))
    Ss.append(np.array([1.0], dtype=float))
    return Bs, Ss

Bs_hf, Ss = product_mps_Bs_from_bitstring(hf_bits)
hf_psi = mps.MPS(Bs=Bs_hf, Ss=Ss, bc="finite")


## 2. Qubit Hamiltonian and coefficient-decoupled MPO

Left/right symbolic Pauli-fragment graphs and the coefficient bridge $\mathbf{M}$ across the
central cut (paper Sec. III and Appendix A, Algorithm 1).


In [ ]:
n_qubits = int(out["n_qubits"])
Ha = ham.from_openfermion(out["qubit_h"], n_qubits)

if n_qubits < 11:  # exact-diagonalization check, small systems only
    ee, ev = np.linalg.eigh(Ha.to_dense())
    print(ee[:10])


In [ ]:
paths_l = gb.get_left_G(list(Ha.terms), n_qubits)
print("***"*20)
paths_r = gb.get_right_G(list(Ha.terms), n_qubits)

In [ ]:
M, Msym = gb.compute_lambda_coefs_matrix(graph_L=paths_l, graph_R=paths_r, H=list(Ha.terms), num_total_sites=n_qubits, coefsym_dict=Ha.term_indices, coefs_dict=Ha.as_dict)

In [ ]:
cut_loc = int(n_qubits / 2)

G_l, pos_l, labels_l = gb.make_graph(paths_l, list(Ha.terms), 0, cut_loc)
G_r, pos_r, labels_r = gb.make_graph(paths_r, list(Ha.terms), 0, cut_loc)


### Symbolic bipartition graphs (paper Fig. 2)

Left/right deduplicated Pauli-fragment graphs on either side of the cut. (Paper Fig. 2 shows
these for H$_2$ and an H$_8$ chain — switch `MOLECULE_KEY` to regenerate those panels.)


In [ ]:
from __future__ import annotations

from typing import Dict, Tuple, Hashable, Optional, Any
import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.lines import Line2D
import numpy as np


def draw_pauli_labeled_graph(
    G: nx.Graph,
    pos: Dict[Hashable, Tuple[float, float]],
    edge_labels: Optional[Dict[Tuple[Hashable, Hashable], str]] = None,
    *,
    # node styling
    with_node_labels: bool = True,
    node_labels: Optional[Dict[Hashable, str]] = None,
    node_color: Any = "pink",
    node_edge_color: Any = "pink",
    node_size: int = 100,
    node_linewidths: float = 0.6,
    font_size: int = 8,
    font_family: str = "DejaVu Serif",
    edge_label_font_size: int = 5,

    # edge styling
    with_edge_labels: bool = True,
    edge_width: float = 1.0,
    edge_alpha: float = 0.95,
    default_edge_color: str = "0.65",
    arrowstyle: str = "-|>",
    arrowsize: int = 10,
    min_source_margin: float = 0.0,
    min_target_margin: float = 0.0,
    connectionstyle: str = "arc3,rad=0.0",

    # edge-label placement (offset)
    edge_label_pos: float = 0.62,          # 0..1 along edge
    edge_label_offset: float = 0.00,     # perpendicular offset (axis-fraction; tune)
    edge_label_jitter: bool = True,       # alternate side (+/-) to reduce overlap
    edge_label_bbox_alpha: float = 0.58,  # label background transparency

    # mapping label -> color
    label2color: Optional[Dict[str, str]] = None,

    # legend & figure
    with_legend: bool = True,
    legend_loc: str = "upper left",
    figsize: Tuple[float, float] = (6.5, 4.5),
    dpi: int = 100,
    ax: Optional[plt.Axes] = None,
):
    """
    Draw a graph with edge colors determined by Pauli-like edge labels,
    and draw edge labels with a perpendicular offset to reduce collisions.

    Edge-label offset behavior:
    - Labels are placed at a point along the edge (edge_label_pos)
    - Then shifted by a normal vector (perpendicular to the edge direction)
      with magnitude edge_label_offset (interpreted as a fraction of axis span)
    - If edge_label_jitter=True, labels alternate sides of the edge
    """
    if label2color is None:
        label2color = {"I": "gray", "X": "red", "Y": "green", "Z": "blue"}

    if edge_labels is None:
        edge_labels = {}

    # Create axes/figure
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    else:
        fig = ax.figure

    # Typography for paper
    plt.rcParams.update({
        "font.family": font_family,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })

    # Determine consistent edge order
    edgelist = list(G.edges())

    # Resolve label for each edge in edgelist (handle (v,u) key if needed)
    resolved = {}
    edge_colors = []
    for u, v in edgelist:
        lab = edge_labels.get((u, v), edge_labels.get((v, u), None))
        resolved[(u, v)] = lab
        edge_colors.append(label2color.get(lab, default_edge_color))

    # Draw nodes
    nx.draw_networkx_nodes(
        G, pos, ax=ax,
        node_size=node_size,
        node_color=node_color,
        edgecolors=node_edge_color,
        linewidths=node_linewidths,
    )

    # Draw edges (directed graphs get arrows)
    is_directed = G.is_directed()
    edge_kwargs = dict(
        ax=ax,
        edgelist=edgelist,
        edge_color=edge_colors,
        width=edge_width,
        alpha=edge_alpha,
        arrows=is_directed,
        arrowstyle=arrowstyle if is_directed else "-",
        arrowsize=arrowsize if is_directed else 10,
        connectionstyle=connectionstyle,
        node_size=node_size,  # IMPORTANT for arrow shrink vs nodes
    )

    try:
        nx.draw_networkx_edges(
            G, pos,
            min_source_margin=min_source_margin,
            min_target_margin=min_target_margin,
            **edge_kwargs
        )
    except TypeError:
        nx.draw_networkx_edges(G, pos, **edge_kwargs)

    # Node labels
    if with_node_labels:
        if node_labels is None:
            node_labels = {n: str(n) for n in G.nodes()}
        nx.draw_networkx_labels(
            G, pos, ax=ax,
            labels=node_labels,
            font_size=font_size,
            font_color="black",
        )

    # Edge labels with offset (manual)
    if with_edge_labels and edge_labels:
        # Convert an axis-fraction offset into data units (so it scales nicely)
        x0, x1 = ax.get_xlim()
        y0, y1 = ax.get_ylim()
        dx_data = (x1 - x0) * edge_label_offset
        dy_data = (y1 - y0) * edge_label_offset

        for k, (u, v) in enumerate(edgelist):
            lab = resolved.get((u, v), None)
            if lab is None:
                continue

            (xu, yu) = pos[u]
            (xv, yv) = pos[v]

            # Base point along edge
            t = float(edge_label_pos)
            mx = (1 - t) * xu + t * xv
            my = (1 - t) * yu + t * yv

            # Normal direction (perpendicular)
            vx, vy = (xv - xu), (yv - yu)
            nrm = np.hypot(vx, vy)
            if nrm == 0:
                continue
            nxn, nyn = (-vy / nrm), (vx / nrm)

            # Alternate side to reduce overlaps
            sgn = 1.0
            if edge_label_jitter:
                sgn = 1.0 if (k % 2 == 0) else -1.0

            # Offset in data units
            lx = mx + sgn * nxn * dx_data
            ly = my + sgn * nyn * dy_data

            ax.text(
                lx, ly, str(lab),
                fontsize=edge_label_font_size,
                ha="center", va="center",
                color="black",
                bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none", alpha=edge_label_bbox_alpha),
                zorder=10
            )

    # Legend
    if with_legend:
        handles = [
            Line2D([0], [0], color=label2color[k], lw=3.0, label=k)
            for k in ["I", "X", "Y", "Z"] if k in label2color
        ]
        if handles:
            ax.legend(
                handles=handles,
                loc=legend_loc,
                frameon=True,
                framealpha=0.9,
                fancybox=True,
                borderpad=0.5,
                handlelength=2.0,
                fontsize=max(9, font_size - 1),
            )

    ax.set_axis_off()
    #ax.margins(0.06)
    fig.tight_layout()
    plt.tight_layout()
    return fig, ax


In [ ]:
fig, ax = draw_pauli_labeled_graph(
    G_l, pos_l, labels_l,
    with_node_labels=False,     # color-only edges for paper clarity
    node_color="gold",
    with_edge_labels=True,     # no text, just edge color
    legend_loc=(0,0.5),
    figsize=(6., 4.),
    with_legend=False
)

#fig.savefig("figs/pauli_graph.pdf", bbox_inches="tight", transparent=True)   # best for papers
#fig.savefig("figs/pauli_graph.png", dpi=100, bbox_inches="tight")

fig, ax = draw_pauli_labeled_graph(
    G_r, pos_r, labels_r,
    with_node_labels=False,     # color-only edges for paper clarity
    node_color="gold",
    with_edge_labels=True,     # no text, just edge color
    legend_loc=(0,0.5),
    figsize=(5., 4.),
    with_legend=False
)

#fig.savefig("figs/pauli_graph_r.pdf", bbox_inches="tight", transparent=True)   # best for papers
#fig.savefig("figs/pauli_graph_r.png", dpi=100, bbox_inches="tight")


### Hamiltonian MPO

In [ ]:
t0 = time.perf_counter()
wmpo, norm_mpo, discarded = get_mpo(
    N_qubit=n_qubits, sys_Ham=Ha, epsilon=MPO_EPS, chi_max=MPO_CHI_MAX, show_info=True
)
t1 = time.perf_counter()

print(f"MPO build time: {t1 - t0:.2f} s | discarded terms: {discarded}")
print("bond dimensions:", bonds_my(wmpo))


## 3. DMRG reference state (pretrained MPS)

Two-site DMRG on the Hamiltonian MPO. Settings come from the configuration cell;
the paper uses $\chi = 8$ and $\chi = 32$ reference states.


In [ ]:
from qu_alg_qu_sim_tensor_networks import mps_helper, mps, dmrg
chi_dmrg = CHI_DMRG
def _dmrg_ground_state_from_mpo(
    wmpo,
    *,
    chi_max: int = chi_dmrg,
    eps: float = DMRG_EPS,
    sweeps: int = DMRG_SWEEPS,
    init_chi: int = DMRG_INIT_CHI,
    seed: int = DMRG_SEED,
    verbose: bool = True,
):
    """
    Run 1D DMRG on an MPO using your mps_helper / dmrg stack.

    Returns
    -------
    psi_mps : MPS
        Ground state as MPS.
    E0 : float
        Lowest energy (real).
    """
    import copy
    psi = copy.deepcopy(hf_psi)
    model = mps_helper.vvdd_ddvv_mpo_swap(wmpo)
    psi = mps.init_random_bond_chi_MPS(model.L, model.d, init_chi, model.bc, seed=seed)
    #psi = hf_psi
    eng = dmrg.DMRGEngine(psi, model, chi_max=chi_max, eps=eps)
    E_prev = None
    for it in range(sweeps):
        eng.sweep()
        E = mps_helper.expectation_value(wmpo, psi)
        if verbose:
            print(f"[DMRG] sweep {it:02d}: E = {E.real:.10f}, chi = {psi.get_chi()}")
        if E_prev is not None:
            rel = abs((E - E_prev) / (E if E != 0 else 1.0))
            if rel.real < 1e-11:
                if verbose:
                    print(f"[DMRG] converged at sweep {it:02d}")
                break
        E_prev = E
    return _mps_to_numpy(psi), float(E_prev.real)


def _mps_to_numpy(psi):
    """Move MPS tensors back to host memory (no-op on CPU backends).

    The sampler and bridge optimizer are NumPy-only, so after a GPU (CuPy)
    DMRG run the state must be converted before further processing.
    """
    try:
        import cupy as cp
        psi.Bs = [cp.asnumpy(B) if isinstance(B, cp.ndarray) else np.asarray(B) for B in psi.Bs]
        psi.Ss = [cp.asnumpy(S) if isinstance(S, cp.ndarray) else np.asarray(S) for S in psi.Ss]
    except ImportError:
        pass
    return psi


psi, E_dmrg = _dmrg_ground_state_from_mpo(wmpo)

E_dmrg

## 4. Perfect sampling of Pauli strings from the pretrained MPS

The sampling distribution (referee item: define $\Pi(P)$ explicitly) is

$$\Pi(P) \;=\; \frac{\langle\psi|P|\psi\rangle^{2}}{2^{N}},$$

sampled *exactly* (no Markov chain) by conditional sweeps over the MPS transfer operators —
paper Appendix F, Algorithm 2 (`estimate_magic_perfect_from_mps`). The seed is recorded in the
configuration cell.


In [ ]:
# Pauli matrices, stacked in the order [I, X, Y, Z]
I = np.array([[1, 0], [0, 1]], dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
ops = np.stack([I, X, Y, Z], axis=0)  # (4,2,2)

# Perfect sampling of Pauli strings P with probability Pi(P) = <psi|P|psi>^2 / 2^N
res_samples = mps_helper.estimate_magic_perfect_from_mps(
    psi.Bs, ops, n_samples=N_PAULI_SAMPLES, seed=SAMPLING_SEED
)
print("unique sampled strings:", len(res_samples["samples"]))


### Polar map of sampled Pauli strings (paper Fig. 4)

Radial coordinate = normalized Pauli weight, angle = bitmask hash of the operator support,
dot area $\propto \sqrt{\Pi(P)}$. Classified by excitation order (diagonal / single / double / higher).


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl

mpl.rcParams.update({
    "font.family":      "serif",
    "font.serif":       ["STIXGeneral", "DejaVu Serif", "Times"],
    "mathtext.fontset": "stix",
})

raw_data = res_samples["samples"]

# ── classify ──────────────────────────────────────────────────────────────────
COLORS = {
    'diag':   '#bfdbfe',
    'single': '#f59e0b',
    'double': '#ef4444',
    'other':  '#a78bfa',
}

items = []
for pauli, coeff in raw_data.items():
    xy     = pauli.count('X') + pauli.count('Y')
    cls    = {0:'diag', 2:'single', 4:'double'}.get(xy, 'other')
    weight = sum(1 for c in pauli if c != 'I')
    items.append(dict(p=pauli, coeff=coeff, cls=cls, weight=weight, xy=xy))

max_w  = max(it['weight'] for it in items)
coeffs = [it['coeff'] for it in items]
c_min, c_max = min(coeffs), max(coeffs)
counts = {cls: sum(1 for it in items if it['cls'] == cls) for cls in COLORS}

# ── bitmask angle ─────────────────────────────────────────────────────────────
def bitmask_angle(pauli: str, ops: str) -> float:
    bits = 0
    for i, c in enumerate(pauli):
        if c in ops:
            bits |= (1 << i)
    bits &= 0xFFFFFFFF
    bits  = (~bits + (bits << 21)) & 0xFFFFFFFF
    bits ^= bits >> 24
    bits  = (bits + (bits << 3))   & 0xFFFFFFFF
    bits ^= bits >> 14
    bits  = (bits * 0x265E5A51)    & 0xFFFFFFFF
    bits ^= bits >> 28
    bits  = (bits + (bits << 31))  & 0xFFFFFFFF
    return (bits / 0xFFFFFFFF) * 2 * np.pi

def dot_radius(coeff):
    norm = np.sqrt((coeff - c_min) / (c_max - c_min + 1e-30))
    return 18 + norm * 280

def djitter(pauli: str, scale: float):
    h = 5381
    for c in pauli:
        h = ((h * 33) ^ ord(c)) & 0xFFFFFFFF
    a = ((h & 0xFFFF) / 0xFFFF - 0.5) * 2
    b = (((h >> 16) & 0xFFFF) / 0xFFFF - 0.5) * 2
    return a * scale, b * scale

JITTER_SCALE = 0.018
USE_JITTER   = True

for it in items:
    r = it['weight'] / max_w
    if it['weight'] == 0:
        angle = 0.0
    elif it['cls'] == 'diag':
        angle = bitmask_angle(it['p'], 'Z')
    else:
        angle = bitmask_angle(it['p'], 'XY')
    jx, jy = djitter(it['p'], JITTER_SCALE) if USE_JITTER else (0, 0)
    it['x'] = r * np.cos(angle) + jx
    it['y'] = r * np.sin(angle) + jy
    it['r'] = r
    it['angle'] = angle

# ── figure: use matplotlib native polar projection ───────────────────────────
fig = plt.figure(figsize=(6, 6), dpi=130)
fig.patch.set_facecolor('white')

# main polar axis
ax = fig.add_subplot(111, projection='polar')
ax.set_facecolor('white')

# ── polar grid styling ────────────────────────────────────────────────────────
# angle grid: every π/6
ax.set_thetagrids(
    np.arange(0, 360, 30),
    labels=[r'$0$', r'$\pi/6$', r'$\pi/3$', r'$\pi/2$',
            r'$2\pi/3$', r'$5\pi/6$', r'$\pi$',
            r'$7\pi/6$', r'$4\pi/3$', r'$3\pi/2$',
            r'$5\pi/3$', r'$11\pi/6$'],
    fontsize=11
)

# radial grid: at even weights only
r_ticks = [w / max_w for w in range(2, max_w + 1, 2)]
r_labels = [f'$w={w}$' for w in range(2, max_w + 1, 2)]
ax.set_rticks(r_ticks)
ax.set_yticklabels(r_labels, fontsize=7, color='#64748b')
ax.set_rlabel_position(-22)          # move radial labels off the spokes

# extra fine rings at odd weights (no label)
for w in range(1, max_w + 1, 2):
    r = w / max_w
    ax.plot(np.linspace(0, 2*np.pi, 360), [r]*360,
            color='#94a3b8', lw=0.35, alpha=0.25, zorder=0)

# grid line styling
ax.grid(color='#94a3b8', linewidth=0.5, alpha=0.35)
ax.spines['polar'].set_color('#94a3b8')
ax.spines['polar'].set_linewidth(0.8)

# ── dots ──────────────────────────────────────────────────────────────────────
for cls in ('diag', 'other', 'single', 'double'):
    sub = [it for it in items if it['cls'] == cls]
    if not sub:
        continue
    angles = [it['angle'] for it in sub]
    rs     = [it['r']     for it in sub]
    ss     = [dot_radius(it['coeff']) for it in sub]
    alpha  = 0.55 if cls == 'diag' else 0.78
    ec     = '#93c5fd' if cls == 'diag' else 'white'
    ax.scatter(angles, rs, s=ss, c=COLORS[cls], alpha=alpha,
               edgecolors=ec, linewidths=0.4, zorder=3)

# ── identity at center ────────────────────────────────────────────────────────
ax.scatter([0], [0], s=80, facecolors='none',
           edgecolors='#64748b', linewidths=1.5, zorder=5)

# ── legend ────────────────────────────────────────────────────────────────────
legend_items = [
    ('identity',                               dict(marker='o', markerfacecolor='none',
                                                    markeredgecolor='#64748b', markeredgewidth=1.5)),
    (f'diagonal  $n={counts["diag"]}$',   dict(marker='o', color='#bfdbfe')),
    (f'single excitation  $n={counts["single"]}$', dict(marker='o', color='#f59e0b')),
    (f'double excitation  $n={counts["double"]}$', dict(marker='o', color='#ef4444')),
    (f'higher excitation  $n={counts["other"]}$',  dict(marker='o', color='#a78bfa')),
]
handles = [plt.Line2D([0], [0], linestyle='none', markersize=7, **kw)
           for _, kw in legend_items]
labels  = [lbl for lbl, _ in legend_items]
ax.legend(handles, labels,
          loc='upper center',
          bbox_to_anchor=(0.5, 1.23),
          ncol=3,
          framealpha=0.0,
          fontsize=10.5,
          prop={'size': 10.5},
          borderpad=0.6,
          columnspacing=0.4,
          handletextpad=0.4,
          )

plt.savefig(os.path.join(FIGS_DIR, f'pauli_map_polar_{MOLECULE_KEY}_chi_{chi_dmrg}.pdf'),
            bbox_inches='tight', facecolor='white')
plt.show()

## 5. Pool curation

Explicit curation rule (referee item 5.2-4): keep **all** strings containing at least one
$X$/$Y$ (excitation strings), plus only the `TOP_K_DIAGONAL` purely diagonal ($I/Z$-only)
strings ranked by $|$coefficient$|$; discard everything below a numerical threshold.


In [ ]:
def filter_pauli_excitations(samples, top_k_diagonal=5, threshold=1e-12):
    """
    Filters Pauli strings to keep useful quantum excitations (X/Y) 
    and only the most significant classical configurations (Z/I).

    Args:
        samples (dict): Dictionary of {Pauli_string: coefficient}.
        top_k_diagonal (int): Number of Z/I-only strings to keep (sorted by magnitude).
        threshold (float): Numerical cutoff to discard absolute zeros or noise.

    Returns:
        dict: The filtered dictionary containing all X/Y terms and top_k Z terms.
    """
    
    diagonal_terms = []      # List for Z/I only strings
    excitation_terms = {}    # Dict for X/Y strings
    
    # 1. Classify and Filter Noise
    for pauli, coeff in samples.items():
        # Skip numerical noise (optional but recommended)
        if abs(coeff) < threshold:
            continue
            
        # Check if the string contains any 'Quantum' operators (X or Y)
        if 'X' in pauli or 'Y' in pauli:
            # Keep ALL excitations (as requested)
            excitation_terms[pauli] = coeff
        else:
            # Store Diagonal terms for sorting later
            diagonal_terms.append((pauli, coeff))
    
    # 2. Sort Diagonal Terms by Magnitude (Largest first)
    # We use abs(coeff) for sorting to find the most significant weights
    diagonal_terms.sort(key=lambda x: abs(x[1]), reverse=True)
    
    # 3. Keep only the top K diagonal terms
    kept_diagonal = dict(diagonal_terms[:top_k_diagonal])
    
    # 4. Merge back together
    final_ansatz = {**kept_diagonal, **excitation_terms}
    
    print(f"--- Filtering Report ---")
    print(f"Original Count:   {len(samples)}")
    print(f"Excitations (X/Y) Kept: {len(excitation_terms)}")
    print(f"Diagonals (Z/I) Kept:   {len(kept_diagonal)} (Top {top_k_diagonal})")
    print(f"Final Ansatz Size: {len(final_ansatz)}")
    
    return final_ansatz


optimized_ansatz = filter_pauli_excitations(res_samples["samples"], top_k_diagonal=5)
#optimized_ansatz = filter_pauli_excitations(filter_samples, top_k_diagonal=5)

In [ ]:
optimized_ansatz = filter_pauli_excitations(
    res_samples["samples"], top_k_diagonal=TOP_K_DIAGONAL
)
# include the identity string in the pool (its weight is re-optimized by the bridge step)
optimized_ansatz["I" * n_qubits] = 0.9
print("curated pool size:", len(optimized_ansatz))


## 6. Bridge-coefficient optimization (Option C)

The curated, state-adapted pool becomes the generator $\hat G(\vec\theta)$: fixed symbolic
left/right fragments, tunable bridge $\mathbf{S}_g$. The bridge is optimized by an adaptive
generalized eigenproblem $H_\mathrm{eff}\,v = E\,N_\mathrm{eff}\,v$ restricted to an actively
grown pair subspace (dense ED or LOBPCG).


In [ ]:
# Generator pool Gs = curated MPS-sampled pool (paper Sec. VI workflow)
Gs = ham.Hamiltonian(n_qubits=out["n_qubits"])
Gs.terms = dict(optimized_ansatz)
print("generator pool terms:", len(Gs.terms))


In [ ]:
# (optional inspection) symbolic graphs and bridge matrix of the generator pool
pathsg_l = gb.get_left_G(list(Gs.terms), n_qubits)
pathsg_r = gb.get_right_G(list(Gs.terms), n_qubits)

Mg, Mgsym = gb.compute_lambda_coefs_matrix(
    graph_L=pathsg_l, graph_R=pathsg_r, H=list(Gs.terms),
    num_total_sites=n_qubits, coefsym_dict=Gs.term_indices, coefs_dict=Gs.as_dict
)
print("bridge shape:", np.asarray(Mg).shape)


In [ ]:
# ============================================================
# FULL SCRIPT: Adaptive Option-C with robust AUTO inner solve
#   - Dense ED (generalized EVP) if K <= dense_threshold
#   - Robust to non-PD metric via shift + QZ fallback
#   - LOBPCG wrapped with non-PD fallback to dense ED
#   - Convergence plotting (+ E_dmrg black horizontal line)
#   - NEW: per-outer/per-inner iteration history as nested dict
# ============================================================
"""
Adaptive Option-C bridge optimization for an MPO cut.

This module solves a generalized eigenvalue problem for a bridge tensor S_g
defined across a fixed bipartition of an MPO/MPS network. The effective
Hamiltonian and metric are never formed explicitly in the full bridge space;
instead, they are represented as LinearOperators built from left/right cut
environments.

Main strategy
-------------
1. Build full-space effective operators H_eff and N_eff acting on the bridge.
2. Restrict the problem to a reduced subspace indexed by selected pair labels
   (i, j), using basis maps derived from A_left and A_right.
3. Solve the reduced generalized eigenvalue problem
       H_sub alpha = E N_sub alpha
   using either dense ED or LOBPCG.
4. Lift the reduced solution back to the full bridge space and evaluate the
   full residual
       r = H_eff v - E N_eff v.
5. Express the residual in the old pair basis and add the largest missing
   pair components to the active subspace.
6. Repeat until convergence, mask saturation, or a maximum subspace size
   is reached.

This is therefore a residual-driven adaptive subspace expansion algorithm for
bridge optimization.

Key controls
------------
outer_rounds : int
    Maximum number of adaptive refinement rounds.

add_per_round : int
    Number of new pair indices (i, j) added after each outer round, chosen
    from the largest residual components not already present in the mask.

dense_threshold : int
    If the reduced subspace size K is at most this value and inner_method is
    "auto", solve the generalized eigenproblem by dense ED. Otherwise use
    LOBPCG.

history
-------
The optimizer records both flat convergence arrays and nested per-round solver
diagnostics in history["iters"].
"""

import numpy as np
import scipy.linalg
from scipy.sparse.linalg import LinearOperator, lobpcg
import matplotlib.pyplot as plt
import warnings
import time

# ------------------ helpers ------------------
def squeeze_env5_to3(env5):  # (1,bp,a,b,1)->(bp,a,b)
    return env5[0, :, :, :, 0]

def ortho_qr(V, tol=1e-12):
    if V.size == 0:
        return V
    Q, R = np.linalg.qr(V)
    keep = np.abs(np.diag(R)) > tol
    if keep.ndim == 0:
        keep = np.array([bool(keep)])
    return Q[:, keep]

def rayleigh(Hop, Nop, v):
    Hv = Hop @ v
    Nv = Nop @ v
    den = np.vdot(v, Nv)
    if abs(den) < 1e-30:
        return np.vdot(v, Hv)
    return np.vdot(v, Hv) / den

def identity_mpo_tensors(n):
    W = np.eye(2, dtype=np.complex128).reshape(2, 2, 1, 1)
    return [W] * n

# ------------------ low-rank mid only ------------------
def lowrank_factorize_bridge(S, r):
    U, s, Vh = np.linalg.svd(S, full_matrices=False)
    r = min(int(r), len(s))
    return U[:, :r], s[:r], Vh[:r, :]

def make_effective_linop_lowrank(Lcut5, Rcut5, S_mid, r, reg_eps=0.0):
    L = squeeze_env5_to3(Lcut5)  # (bpL,aL,bL)
    R = squeeze_env5_to3(Rcut5)  # (bpR,aR,bR)
    bpL, aL, bL = L.shape
    bpR, aR, bR = R.shape
    if S_mid.shape != (aL, aR):
        raise ValueError(f"S_mid shape {S_mid.shape} != ({aL},{aR})")

    U, s, Vh = lowrank_factorize_bridge(S_mid, r)
    Lsharp = np.einsum("pab,ar->prb", L, U, optimize=True)      # (bpL,r,bL)
    Rsharp = np.einsum("rA,qAB->qrB", Vh, R, optimize=True)     # (bpR,r,bR)

    n = bpL * bpR
    if n != bL * bR:
        raise ValueError(f"Need square: bpL*bpR={n} vs bL*bR={bL*bR}")

    def mv(x):
        x = np.asarray(x, dtype=np.complex128)
        X = x.reshape(bL, bR)
        Y = np.einsum("bB,qrB->bqr", X, Rsharp, optimize=True)
        Z = np.einsum("prb,bqr->pqr", Lsharp, Y, optimize=True)
        out = np.einsum("pqr,r->pq", Z, s, optimize=True).reshape(-1)
        return out + reg_eps * x if reg_eps else out

    return LinearOperator((n, n), matvec=mv, dtype=np.complex128), (bpL, bpR)

# ------------------ Option-C mask ops ------------------
def mask_pairs_from_seed(S_old, nonzero_tol=1e-14):
    S_old = np.asarray(S_old)
    nz = np.abs(np.nan_to_num(S_old, nan=0.0)) > nonzero_tol
    nn = np.isnan(S_old)
    M = nz | nn
    I, J = np.where(M)
    return list(zip(I.tolist(), J.tolist()))

def make_B_ops(A_left, A_right, pairs):
    if len(pairs) == 0:
        raise ValueError("empty subspace mask (pairs)")

    I = np.array([i for (i, j) in pairs], dtype=np.int64)
    J = np.array([j for (i, j) in pairs], dtype=np.int64)

    Ucols = A_left[:, I]        # (bpL_new, K)
    Vrows = A_right[J, :]       # (K, bpR_new)
    bpL_new, K = Ucols.shape
    bpR_new = Vrows.shape[1]

    def B_to_full(alpha):
        alpha = np.asarray(alpha, dtype=np.complex128).reshape(K)
        X = Ucols @ (alpha[:, None] * Vrows)     # (bpL_new,bpR_new)
        return X.reshape(-1)

    def BT_to_sub(vec):
        Y = np.asarray(vec, dtype=np.complex128).reshape(bpL_new, bpR_new)
        tmp = Ucols.conj().T @ Y                 # (K,bpR_new)
        return np.einsum("kr,kr->k", tmp, Vrows.conj(), optimize=True)

    return B_to_full, BT_to_sub, (bpL_new, bpR_new), K

def restrict_op_to_subspace(op_full, B_to_full, BT_to_sub, K):
    def mv(alpha):
        v = B_to_full(alpha)
        w = op_full @ v
        return BT_to_sub(w)
    return LinearOperator((K, K), matvec=mv, dtype=np.complex128)

def lift_alpha_to_new_pairs(alpha_old, pairs_old, pairs_new, *, fallback_first=True):
    idx_old = {p: i for i, p in enumerate(pairs_old)}
    Knew = len(pairs_new)
    alpha_new = np.zeros(Knew, dtype=np.complex128)
    for j, p in enumerate(pairs_new):
        i = idx_old.get(p, None)
        if i is not None:
            alpha_new[j] = alpha_old[i]
    if np.linalg.norm(alpha_new) < 1e-30 and fallback_first:
        alpha_new[0] = 1.0
    return alpha_new

def add_pairs_from_matrix_abs(M_old, pairs, add=1000):
    pairs_set = set(pairs)
    score = np.abs(M_old).copy()

    for (i, j) in pairs_set:
        if 0 <= i < score.shape[0] and 0 <= j < score.shape[1]:
            score[i, j] = 0.0

    flat = score.ravel()
    if flat.size == 0:
        return pairs, 0
    k = min(int(add), flat.size)
    idx = np.argpartition(flat, -k)[-k:]
    idx = idx[np.argsort(flat[idx])[::-1]]

    nR = score.shape[1]
    new_pairs = []
    for t in idx:
        i = int(t // nR); j = int(t % nR)
        if (i, j) in pairs_set:
            continue
        new_pairs.append((i, j))
        pairs_set.add((i, j))
        if len(new_pairs) >= add:
            break
    return pairs + new_pairs, len(new_pairs)

def make_lobpcg_init(alpha0, K, block, seed=0):
    rng = np.random.default_rng(seed)
    b = max(2, int(block))
    b = min(b, max(2, K // 4))
    b = min(b, K)
    X = np.zeros((K, b), dtype=np.complex128)

    a0 = np.asarray(alpha0, dtype=np.complex128).reshape(K)
    a0 = a0 / (np.linalg.norm(a0) + 1e-30)
    X[:, 0] = a0
    for j in range(1, b):
        z = rng.normal(size=K) + 1j * rng.normal(size=K)
        z = z / (np.linalg.norm(z) + 1e-30)
        X[:, j] = z
    X = ortho_qr(X)
    return X

# ------------------ Dense ED utilities ------------------
def linop_to_dense(op, K):
    """Materialize (K,K) dense matrix from a LinearOperator via matvecs."""
    A = np.empty((K, K), dtype=np.complex128)
    ej = np.zeros(K, dtype=np.complex128)
    for j in range(K):
        ej[:] = 0
        ej[j] = 1.0
        A[:, j] = op @ ej
    return A

def solve_gep_dense_ed_robust(
    Hop, Nop, K, *,
    symmetrize=True,
    pd_tol=1e-12,
    max_shift_tries=8,
    shift_growth=10.0,
    fallback_qz=True,
):
    """
    Robust smallest generalized eigenpair for Hop x = E Nop x (dense).
    - Hermitize H,N
    - Try to enforce N > 0 by diagonal shifts until eigh works
    - If not possible, QZ fallback eig(H,N)
    Returns (E0, alpha).

    NEW: writes per-call info dict to solve_gep_dense_ed_robust.last_info
    """
    info = {
        "method": "dense_ed",
        "K": int(K),
        "symmetrize": bool(symmetrize),
        "pd_tol": float(pd_tol),
        "max_shift_tries": int(max_shift_tries),
        "shift_growth": float(shift_growth),
        "fallback_qz": bool(fallback_qz),
        "attempts": [],
    }

    H = linop_to_dense(Hop, K)
    N = linop_to_dense(Nop, K)

    if symmetrize:
        H = 0.5 * (H + H.conj().T)
        N = 0.5 * (N + N.conj().T)

    # scale-aware PD target
    diag_scale = float(max(1.0, np.max(np.abs(np.diag(N))).real))
    target = pd_tol * diag_scale
    info["diag_scale"] = diag_scale
    info["pd_target"] = target

    shift = 0.0
    for attempt in range(max_shift_tries + 1):
        attempt_rec = {"attempt": int(attempt), "shift": float(shift)}
        try:
            Ns = N + shift * np.eye(K, dtype=np.complex128) if shift != 0.0 else N
            evals, evecs = scipy.linalg.eigh(H, Ns)
            E0 = float(np.real(evals[0]))
            alpha = evecs[:, 0].astype(np.complex128, copy=False)

            # normalize in ORIGINAL N (not Ns)
            den = np.vdot(alpha, N @ alpha)
            if np.abs(den) > 1e-30:
                alpha = alpha / np.sqrt(np.abs(den))

            attempt_rec["status"] = "success"
            attempt_rec["E0"] = float(E0)
            info["attempts"].append(attempt_rec)

            info["status"] = "eigh"
            info["shift_used"] = float(shift)

            solve_gep_dense_ed_robust.last_info = info
            return E0, alpha

        except np.linalg.LinAlgError as e:
            attempt_rec["status"] = "fail"
            attempt_rec["exception"] = repr(e)

            try:
                wN = np.linalg.eigvalsh(N)
                lam_min = float(np.min(wN).real)
            except np.linalg.LinAlgError as e2:
                lam_min = float("-inf")
                attempt_rec["eigvalsh_exception"] = repr(e2)

            attempt_rec["lam_min_N"] = float(lam_min)
            info["attempts"].append(attempt_rec)

            need = 0.0 if lam_min > target else (target - lam_min)
            if shift == 0.0:
                shift = max(need, target) if np.isfinite(need) else target
            else:
                shift = max(shift * shift_growth, need)

    if not fallback_qz:
        info["status"] = "failed_no_qz"
        solve_gep_dense_ed_robust.last_info = info
        raise np.linalg.LinAlgError("Metric not PD: dense generalized eigh failed after shifts.")

    # QZ fallback: works for indefinite/singular N
    w, V = scipy.linalg.eig(H, N)
    finite = np.isfinite(w)
    w = w[finite]
    V = V[:, finite]

    info["status"] = "qz"
    info["n_finite"] = int(w.size)

    if w.size == 0:
        solve_gep_dense_ed_robust.last_info = info
        raise np.linalg.LinAlgError("QZ fallback produced no finite eigenvalues.")

    # pick smallest real eigenvalue among those with positive N-norm
    best = None
    best_key = None
    for i in range(w.size):
        wi = w[i]
        vi = V[:, i]
        den = np.vdot(vi, N @ vi)
        if den.real <= 1e-14:
            continue
        key = (float(wi.real), float(abs(wi.imag)))
        if best_key is None or key < best_key:
            best_key = key
            best = i

    if best is None:
        best = int(np.argmin(w.real))

    E0 = float(np.real(w[best]))
    alpha = V[:, best].astype(np.complex128, copy=False)

    den = np.vdot(alpha, N @ alpha)
    if np.abs(den) > 1e-30:
        alpha = alpha / np.sqrt(np.abs(den))

    info["qz_chosen_index"] = int(best)
    info["qz_E0"] = float(E0)
    info["qz_w_imag"] = float(abs(w[best].imag))

    solve_gep_dense_ed_robust.last_info = info
    return E0, alpha

# store last call info here
solve_gep_dense_ed_robust.last_info = None


# ------------------ Ghost-safe canonical-orthogonalization solver ------------------
def solve_gep_dense_ed_canonical(Hop, Nop, K, *, rcond=1e-10):
    """
    Ghost-free smallest generalized eigenpair for  H x = E N x  (dense).

    Cure for spurious ("ghost") eigenvalues caused by a near-singular / linearly
    dependent metric N: perform canonical (Loewdin) orthogonalization, i.e. keep
    only the strictly-positive part of N,
        N = U diag(s) U^H ,  keep s_k > rcond * max(s),
        X = U_keep diag(1/sqrt(s_keep)),
    then solve the well-conditioned standard problem  (X^H H X) y = E y  in that
    range. By construction the metric is +I there, so no null-space direction can
    produce an unphysical eigenvalue. Returns (E0, alpha) with alpha in the full
    K-space, normalized so that alpha^H N alpha = 1.
    """
    H = linop_to_dense(Hop, K)
    N = linop_to_dense(Nop, K)
    H = 0.5 * (H + H.conj().T)
    N = 0.5 * (N + N.conj().T)

    s, U = np.linalg.eigh(N)                     # ascending eigenvalues of N
    smax = float(s[-1].real) if s.size else 0.0
    keep = s.real > rcond * max(smax, 1.0)
    if not np.any(keep):
        raise np.linalg.LinAlgError("metric N has no positive directions above rcond.")
    Uk = U[:, keep]
    sk = s[keep].real
    X = Uk / np.sqrt(sk)[None, :]                # (K, m) canonical basis
    Htil = X.conj().T @ H @ X
    Htil = 0.5 * (Htil + Htil.conj().T)
    w, Y = np.linalg.eigh(Htil)
    E0 = float(np.real(w[0]))
    alpha = (X @ Y[:, 0]).astype(np.complex128, copy=False)
    den = np.vdot(alpha, N @ alpha)
    if np.abs(den) > 1e-30:
        alpha = alpha / np.sqrt(np.abs(den))
    return E0, alpha


# ------------------ Robust LOBPCG wrapper ------------------
def solve_gep_lobpcg_robust(
    Hop, Nop, K, alpha0, *,
    tol=1e-6, maxiter=200, block=16, seed=0,
    verbose=True,
    dense_fallback_threshold=5000,
):
    """
    LOBPCG with automatic fallback to dense ED if LOBPCG fails.
    Returns (E0, alpha_opt, converged: bool)

    NEW: writes per-call info dict to solve_gep_lobpcg_robust.last_info
         and (if supported by SciPy) includes lambda/residual histories.
    """
    info = {
        "method": "lobpcg",
        "K": int(K),
        "tol": float(tol) if tol is not None else None,
        "maxiter": int(maxiter) if maxiter is not None else None,
        "block": int(block),
        "seed": int(seed),
        "status": "unknown",
        "warnings": [],
    }

    X = make_lobpcg_init(alpha0, K, block=block, seed=seed)

    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")

            # --- try to request history (SciPy-version dependent) ---
            try:
                res = lobpcg(
                    Hop, X, B=Nop, largest=False,
                    tol=tol, maxiter=maxiter,
                    verbosityLevel=(1 if verbose else 0),
                    retLambdaHistory=True,
                    retResidualNormsHistory=True,
                )
                evals, evecs, lam_hist, res_hist = res
                info["lambda_history"] = lam_hist
                info["residual_norm_history"] = res_hist
                info["history_supported"] = True
            except TypeError:
                # fallback: no history available in this SciPy
                evals, evecs = lobpcg(
                    Hop, X, B=Nop, largest=False,
                    tol=tol, maxiter=maxiter,
                    verbosityLevel=(1 if verbose else 0),
                )
                info["lambda_history"] = None
                info["residual_norm_history"] = None
                info["history_supported"] = False

        info["warnings"] = [str(w.message) for w in caught]

        i0 = int(np.argmin(evals))
        E0 = float(np.real(evals[i0]))
        alpha_opt = evecs[:, i0].astype(np.complex128, copy=False)

        converged = True
        for w in caught:
            if "not converged" in str(w.message).lower():
                converged = False

        info["status"] = "ok" if converged else "not_converged"
        info["E0"] = float(E0)
        info["i0"] = int(i0)
        info["converged"] = bool(converged)

        solve_gep_lobpcg_robust.last_info = info
        return E0, alpha_opt, converged

    except Exception as e:
        info["status"] = "failed"
        info["exception"] = repr(e)

        if verbose:
            print(f"[lobpcg] FAILED: {e}")
            print(f"[lobpcg] Falling back to dense ED (K={K})")

        if K > dense_fallback_threshold:
            solve_gep_lobpcg_robust.last_info = info
            raise RuntimeError(
                f"LOBPCG failed and K={K} > dense_fallback_threshold="
                f"{dense_fallback_threshold}. Cannot fall back to dense ED."
            ) from e

        E0, alpha_opt = solve_gep_dense_ed_robust(
            Hop, Nop, K, symmetrize=True
        )

        info["fallback"] = solve_gep_dense_ed_robust.last_info
        info["status"] = "fell_back_to_dense_ed"
        info["E0"] = float(E0)
        info["converged"] = True

        solve_gep_lobpcg_robust.last_info = info
        return E0, alpha_opt, True

# store last call info here
solve_gep_lobpcg_robust.last_info = None


# ============================================================
# Adaptive Option-C with AUTO inner solve + convergence history
# ============================================================
def optimize_bridge_optionC_adaptive(
    *,
    n_qubits, qmpo, mps_hf,
    wgs_L, wgs_R,
    A_left, A_right,
    Sg_old_seed,
    wmpo_H,
    # operators
    rH=256,
    regN=1e-8,
    nonzero_tol=1e-14,
    id_pair_old=(0, 0),
    # inner solver
    tol=1e-6, maxiter=200, block=16, seed=0,
    # outer mask growth
    outer_rounds=8,
    add_per_round=1200,
    max_pairs=50000,
    verbose=True,
    # ED switch
    dense_threshold=1000,
    inner_method="auto",      # "auto" | "ed" | "lobpcg"
    # plotting
    plot_convergence=True,
):
    cut = n_qubits // 2

    # H halves
    wsH_L = wmpo_H.Ms[:cut]
    wsH_R = wmpo_H.Ms[cut:]

    # Identity MPO halves (metric)
    wI = identity_mpo_tensors(n_qubits)
    wsI_L = wI[:cut]
    wsI_R = wI[cut:]

    # Fixed environments
    L_envs_H = qmpo.build_left_envs_Gdag_O_G_tensordot_left_half(
        mps_hf.Bs, wgs_L, wsH_L, cut=cut
    )
    R_envs_H = qmpo.build_right_envs_Gdag_O_G_tensordot_right_half(
        mps_hf.Bs, wgs_R, wsH_R, cut=cut
    )
    L_envs_I = qmpo.build_left_envs_Gdag_O_G_tensordot_left_half(
        mps_hf.Bs, wgs_L, wsI_L, cut=cut
    )
    R_envs_I = qmpo.build_right_envs_Gdag_O_G_tensordot_right_half(
        mps_hf.Bs, wgs_R, wsI_R, cut=cut
    )

    Lcut5_H, Rcut5_H = L_envs_H[cut], R_envs_H[cut]
    Lcut5_I, Rcut5_I = L_envs_I[cut], R_envs_I[cut]

    # mid identities
    _, aL_H, _ = squeeze_env5_to3(Lcut5_H).shape
    _, aR_H, _ = squeeze_env5_to3(Rcut5_H).shape
    if aL_H != aR_H:
        raise ValueError(f"H cut bond mismatch: aL={aL_H} aR={aR_H}")
    S_mid_H = np.eye(aL_H, dtype=np.complex128)

    _, aL_I, _ = squeeze_env5_to3(Lcut5_I).shape
    _, aR_I, _ = squeeze_env5_to3(Rcut5_I).shape
    if aL_I != aR_I:
        raise ValueError(f"I cut bond mismatch: aL={aL_I} aR={aR_I}")
    S_mid_I = np.eye(aL_I, dtype=np.complex128)

    Hop_full, bridge_shape = make_effective_linop_lowrank(
        Lcut5_H, Rcut5_H, S_mid_H, r=min(int(rH), aL_H), reg_eps=0.0
    )
    Nop_full, _ = make_effective_linop_lowrank(
        Lcut5_I, Rcut5_I, S_mid_I, r=int(aL_I), reg_eps=regN
    )

    # initial pairs from seed
    pairs = mask_pairs_from_seed(Sg_old_seed, nonzero_tol=nonzero_tol)
    if id_pair_old not in pairs:
        pairs.append(id_pair_old)

    # warm start alpha0
    alpha0 = np.zeros(len(pairs), dtype=np.complex128)
    alpha0[pairs.index(id_pair_old)] = 1.0

    best = None

    # ---- convergence history ----
    history = {
        "outer_round": [],
        "K": [],
        "E_init": [],
        "E0": [],
        "rnorm": [],
        "method_used": [],
        "wall_time_s": [],
        "n_added": [],
    }

    # --- NEW: structured per-iteration dicts ---
    history["iters"] = {}  # history["iters"][outer] = {"outer": {...}, "inner": {...}}
    history["E_dmrg"] = globals().get("E_dmrg", None)

    for outer in range(int(outer_rounds)):
        t0 = time.perf_counter()

        if len(pairs) >= int(max_pairs):
            if verbose:
                print(f"[outer] reached max_pairs={max_pairs}")
            break

        B_to_full, BT_to_sub, B_shape, K = make_B_ops(A_left, A_right, pairs)
        if tuple(B_shape) != tuple(bridge_shape):
            raise ValueError(
                f"B_shape={B_shape} != bridge_shape={bridge_shape}"
            )

        Hop = restrict_op_to_subspace(Hop_full, B_to_full, BT_to_sub, K)
        Nop = restrict_op_to_subspace(Nop_full, B_to_full, BT_to_sub, K)

        # Rayleigh quotient of current guess
        v0_full = B_to_full(alpha0)
        E_init = float(np.real(rayleigh(Hop_full, Nop_full, v0_full)))

        # -------- inner solve (AUTO) --------
        use_ed = (
            inner_method == "ed"
            or (inner_method == "auto" and K <= int(dense_threshold))
        )

        method_used = "unknown"

        if use_ed:
            method_used = "dense_ed"
            if verbose:
                print(f"[inner] dense ED on K={K} (threshold={dense_threshold})")
            try:
                E0, alpha_opt = solve_gep_dense_ed_robust(
                    Hop, Nop, K, symmetrize=True
                )
            except np.linalg.LinAlgError as e:
                if verbose:
                    print(f"[inner] dense ED failed: {e}. Falling back to LOBPCG.")
                method_used = "lobpcg_after_ed_fail"
                E0, alpha_opt, _ = solve_gep_lobpcg_robust(
                    Hop, Nop, K, alpha0,
                    tol=tol, maxiter=maxiter, block=block,
                    seed=seed + outer, verbose=verbose,
                    dense_fallback_threshold=dense_threshold * 5,
                )
        else:
            method_used = "lobpcg"
            if verbose:
                print(f"[inner] LOBPCG on K={K}")
            E0, alpha_opt, _ = solve_gep_lobpcg_robust(
                Hop, Nop, K, alpha0,
                tol=tol, maxiter=maxiter, block=block,
                seed=seed + outer, verbose=verbose,
                dense_fallback_threshold=dense_threshold * 5,
            )

        # ------------------ ghost-eigenpair guard (any inner solver) ------------------
        # A spurious eigenpair lives in the numerical null space of the metric Nop
        # (linearly-dependent excitations) and yields an unphysical E0 far below the
        # variational floor. Detect it via a near-zero metric density
        #     rho = <alpha, Nop alpha> / <alpha, alpha>
        # and, if found, re-solve with the ghost-free canonical solver. This is a
        # no-op for well-conditioned pools (rho = O(1) >> ghost_tol).
        ghost_tol = 1e-6
        _aa = float(np.vdot(alpha_opt, alpha_opt).real)
        _rho = float(np.vdot(alpha_opt, Nop @ alpha_opt).real) / _aa if _aa > 0 else 0.0
        if (not np.isfinite(E0)) or (_rho < ghost_tol):
            if verbose:
                print(f"[guard] ghost eigenpair (rho={_rho:.2e}, E0={E0:.6f}); "
                      f"re-solving with canonical orthogonalization.")
            E0, alpha_opt = solve_gep_dense_ed_canonical(Hop, Nop, K, rcond=1e-10)
            method_used = method_used + "+canonical"

        # full solution + residual
        v_full = B_to_full(alpha_opt)
        r_full = (Hop_full @ v_full) - E0 * (Nop_full @ v_full)
        rnorm = float(np.linalg.norm(r_full))

        # map residual to OLD basis
        r_mat = r_full.reshape(bridge_shape)
        r_old = A_left.conj().T @ r_mat @ A_right.conj().T

        # grow mask
        pairs_new, n_added = add_pairs_from_matrix_abs(
            r_old, pairs, add=add_per_round
        )
        if id_pair_old not in pairs_new:
            pairs_new.append(id_pair_old)

        dt = time.perf_counter() - t0

        # record history (flat lists)
        history["outer_round"].append(outer)
        history["K"].append(K)
        history["E_init"].append(E_init)
        history["E0"].append(E0)
        history["rnorm"].append(rnorm)
        history["method_used"].append(method_used)
        history["wall_time_s"].append(dt)
        history["n_added"].append(n_added)

        # --- NEW: structured per-outer/per-inner dict record ---
        if method_used.startswith("dense"):
            inner_info = solve_gep_dense_ed_robust.last_info
        else:
            inner_info = solve_gep_lobpcg_robust.last_info

        history["iters"][outer] = {
            "outer": {
                "outer_round": int(outer),
                "K": int(K),
                "E_init": float(E_init),
                "E0": float(E0),
                "rnorm": float(rnorm),
                "method_used": str(method_used),
                "wall_time_s": float(dt),
                "n_added": int(n_added),
            },
            "inner": inner_info,
        }

        if verbose:
            print(
                f"\n[outer {outer:02d}] K={K:6d}  "
                f"E_init={E_init:.12f}  E0={E0:.12f}  "
                f"||r||={rnorm:.3e}  added={n_added}  "
                f"method={method_used}  dt={dt:.2f}s"
            )

        if best is None or E0 < best["E0"]:
            best = dict(
                E0=E0,
                pairs=pairs,
                alpha=alpha_opt,
                v_full=v_full,
                Sg=v_full.reshape(bridge_shape),
                rnorm=rnorm,
            )

        if n_added == 0:
            if verbose:
                print("[outer] mask saturated (no new pairs from residual).")
            break

        alpha0 = lift_alpha_to_new_pairs(alpha_opt, pairs, pairs_new)
        pairs = pairs_new

    best["history"] = history

    # ---- plot convergence ----
    if plot_convergence and len(history["outer_round"]) > 0:
        plot_optionC_convergence(history)

    return best


# ============================================================
# Convergence plotting
# ============================================================
def plot_optionC_convergence(history, save_path=None):
    """Plot energy, residual, subspace size, and timing vs outer iteration."""
    rounds = history["outer_round"]
    if len(rounds) < 1:
        print("[plot] No iterations to plot.")
        return

    fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
    fig.suptitle("Option-C Adaptive Bridge Optimization", fontsize=14, fontweight="bold")

    color_e = "#2563eb"
    color_r = "#dc2626"
    color_k = "#059669"
    color_t = "#7c3aed"

    # --- (0,0): Energy vs iteration ---
    ax = axes[0, 0]
    ax.plot(rounds, history["E0"], "o-", color=color_e, lw=2, markersize=6, label="E₀ (optimized)")
    ax.plot(rounds, history["E_init"], "s--", color=color_e, alpha=0.4, lw=1.2, markersize=5, label="E_init (warm start)")

    # --- NEW: DMRG reference horizontal line (black) ---
    #E_dmrg = history.get("E_dmrg", globals().get("E_dmrg", None))
    if E_dmrg is not None and np.isfinite(E_dmrg):
        ax.axhline(float(E_dmrg), color="black", lw=1.8, alpha=0.9, label="E_DMRG")

    ax.set_xlabel("Outer round")
    ax.set_ylabel("Energy")
    ax.set_title("Energy convergence")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # annotate methods
    for i, r in enumerate(rounds):
        m = history["method_used"][i]
        short = {"dense_ed": "ED", "lobpcg": "LO", "lobpcg_after_ed_fail": "LO*", "unknown": "?"}
        ax.annotate(short.get(m, m[:3]), (r, history["E0"][i]),
                    textcoords="offset points", xytext=(0, 8),
                    fontsize=7, ha="center", color="gray")

    # --- (0,1): Residual norm ---
    ax = axes[0, 1]
    ax.semilogy(rounds, history["rnorm"], "o-", color=color_r, lw=2, markersize=6)
    ax.set_xlabel("Outer round")
    ax.set_ylabel("||residual||")
    ax.set_title("Residual norm")
    ax.grid(True, alpha=0.3, which="both")

    # --- (1,0): Subspace size K ---
    ax = axes[1, 0]
    ax.plot(rounds, history["K"], "o-", color=color_k, lw=2, markersize=6)
    ax2 = ax.twinx()
    ax2.bar(rounds, history["n_added"], alpha=0.25, color=color_k, width=0.6, label="pairs added")
    ax.set_xlabel("Outer round")
    ax.set_ylabel("Subspace size K", color=color_k)
    ax2.set_ylabel("Pairs added", color=color_k, alpha=0.6)
    ax.set_title("Subspace growth")
    ax.grid(True, alpha=0.3)

    # --- (1,1): Wall time ---
    ax = axes[1, 1]
    bars = ax.bar(rounds, history["wall_time_s"], color=color_t, alpha=0.7, width=0.6)
    ax.set_xlabel("Outer round")
    ax.set_ylabel("Wall time (s)")
    ax.set_title("Time per outer round")
    ax.grid(True, alpha=0.3, axis="y")

    # color bars by method
    method_colors = {
        "dense_ed": "#2563eb",
        "lobpcg": "#f59e0b",
        "lobpcg_after_ed_fail": "#dc2626",
    }
    for i, bar in enumerate(bars):
        m = history["method_used"][i]
        bar.set_color(method_colors.get(m, color_t))
        bar.set_alpha(0.7)

    # legend for method colors
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor="#2563eb", alpha=0.7, label="Dense ED"),
        Patch(facecolor="#f59e0b", alpha=0.7, label="LOBPCG"),
        Patch(facecolor="#dc2626", alpha=0.7, label="LOBPCG (ED fallback)"),
    ]
    ax.legend(handles=legend_elements, fontsize=8, loc="upper left")

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"[plot] Saved to {save_path}")
    plt.show()

    return fig


# ============================================================
# HOW TO RUN
# ============================================================
"""
You must already have in your notebook:
  - n_qubits
  - qmpo
  - hf_psi (or mps_hf)
  - wmpo        (Hamiltonian MPO, with .Ms list)
  - gb          (your graph/bridge builder)
  - Gs          (the generator pool object; often: Gs = Ha)
Optionally:
  - E_dmrg      (float) reference energy; will be plotted as black horizontal line
"""

# 1) Build OLD seed mask
wgs_raw, wgs_L_raw, wgs_R_raw, Sg_old_seed, _ = gb.build_mpo_tensors_in_details(
    list(Gs.terms), n_qubits, Gs.as_dict, Gs.term_indices
)

# 2) Build truncated halves + basis maps (A_left, A_right)
wsG, wgs_L, wgs_R, Sg_mid, _, A_left, A_right, _ = gb.build_mpo_tensors_in_details_truncated(
    list(Gs.terms), n_qubits, Gs.as_dict, Gs.term_indices,
    chi_max=360, epsilon=1e-16, normalize_halves=True, verbose=True
)

# 3) Optimize bridge
best = optimize_bridge_optionC_adaptive(
    n_qubits=n_qubits, qmpo=qmpo, mps_hf=hf_psi,
    wgs_L=wgs_L, wgs_R=wgs_R,
    A_left=A_left, A_right=A_right,
    Sg_old_seed=Sg_old_seed,
    wmpo_H=wmpo,                 # Hamiltonian MPO
    rH=256,
    regN=1e-10,                  # try 1e-8 or 1e-7 if still numerically nasty
    tol=1e-3, maxiter=200, block=6,
    outer_rounds=8, add_per_round=1, max_pairs=50000,
    seed=BRIDGE_SEED, verbose=True,
    dense_threshold=1000,        # K <= 1000 -> use dense ED (robust)
    inner_method="lobpcg",       # "auto" / "ed" / "lobpcg"
    plot_convergence=True,       # auto-plot after optimization
)

print("\n==== BEST ====")
print("E0      =", best["E0"])
print("K_best  =", len(best["pairs"]))
print("||r||   =", best["rnorm"])
print("Sg shape=", best["Sg"].shape)

# Access convergence history:
history = best["history"]

# Re-plot later if needed:
plot_optionC_convergence(history,
    save_path=os.path.join(FIGS_DIR, f"convergence_{MOLECULE_KEY}.png"))

# Structured per-iteration dict history:
# history["iters"][outer]["outer"]  -> outer summary dict
# history["iters"][outer]["inner"]  -> solver info dict (dense_ed or lobpcg)

Sg_opt = best["Sg"]


## 7. Energy vs number of sampled strings (paper Fig. 5)

Raw figure data lives in `results/` (referee item: raw numerical data + seeds). The archived
**submission-1** file holds the original single-run numbers. The sweep cell below regenerates
the dataset with **several sampling seeds per point** (`SEED_GRID`); the plot and table then
report **mean ± one standard deviation over seeds** — the repeated-run protocol the referee
asked for. On a laptop the full sweep is hours of compute; in Colab, run it once and download
`results/` afterwards.


In [ ]:
FIG5_DATA_FILE = os.path.join(RESULTS_DIR, "fig5_data_C2H2_sto3g_frozen_submission1.json")

# Submission-1 data (single run per point; energies in Hartree).
# points: [n_excitations_kept, n_samples, optimized_energy]
fig5_submission1 = {
    "molecule": "C2H2_sto3g_frozen",
    "note": "First-submission data; single stochastic run per point, no seed recorded.",
    "chi": {
        "8": {
            "E_dmrg": -75.95523599,
            "points": [
                [15, 50, -75.93680504528422],
                [21, 100, -75.93734567936558],
                [33, 200, -75.93819550781066],
                [49, 300, -75.94132770909269],
                [77, 400, -75.9415143107637],
                [82, 500, -75.94302557477197],
                [113, 750, -75.95099508400051],
                [148, 1000, -75.95490460108614],
            ],
        },
        "32": {
            "E_dmrg": -75.9924042,
            "points": [
                [16, 50, -75.9383479833988],
                [22, 100, -75.94505626003684],
                [37, 200, -75.94971109576261],
                [61, 300, -75.96367615911537],
                [83, 400, -75.96741631493012],
                [97, 500, -75.9674293624036],
                [135, 750, -75.9803293845654],
                [174, 1000, -75.98089350834283],
            ],
        },
    },
}

if not os.path.exists(FIG5_DATA_FILE):
    with open(FIG5_DATA_FILE, "w") as f:
        json.dump(fig5_submission1, f, indent=2)
    print("wrote", FIG5_DATA_FILE)
else:
    print(FIG5_DATA_FILE, "already exists — not overwriting")


In [ ]:
# ----- Full Fig.-5 sweep: DMRG -> sample -> curate -> optimize, per (chi, N_samples, seed) -----
# Expensive. Set RUN_FIG5_SWEEP = True to regenerate the dataset.
RUN_FIG5_SWEEP = False

CHI_GRID       = (8, 32)
N_SAMPLES_GRID = (50, 100, 200, 300, 400, 500, 750, 1000, 10000)
SEED_GRID      = (0, 1, 2, 3, 4)   # 5 sampling seeds per point -> means + uncertainty bands

# Bridge mode:
#   False -> adaptive Option-C: the bridge starts from the MPS-sampled pool and
#            ADDS residual-driven operator components each round (paper default).
#   True  -> samples-only: solve ONCE on the sampled pool, add NO new connections
#            (outer_rounds=1). Uses strictly the Pauli strings sampled from the MPS.
USE_ONLY_MPS_SAMPLES = False


def run_fig5_point(psi_ref, chi_ref, n_smp, smp_seed):
    """One Fig.-5 data point for a given reference MPS, sample count, and seed."""
    res = mps_helper.estimate_magic_perfect_from_mps(
        psi_ref.Bs, ops, n_samples=n_smp, seed=smp_seed)
    pool_c = filter_pauli_excitations(res["samples"], top_k_diagonal=TOP_K_DIAGONAL)
    pool_c["I" * n_qubits] = 0.9

    Gp = ham.Hamiltonian(n_qubits=n_qubits)
    Gp.terms = dict(pool_c)

    _, _, _, Sg_seed_p, _ = gb.build_mpo_tensors_in_details(
        list(Gp.terms), n_qubits, Gp.as_dict, Gp.term_indices)
    _, wgs_L_p, wgs_R_p, _, _, A_left_p, A_right_p, _ = gb.build_mpo_tensors_in_details_truncated(
        list(Gp.terms), n_qubits, Gp.as_dict, Gp.term_indices,
        chi_max=360, epsilon=1e-16, normalize_halves=True, verbose=False)

    best_p = optimize_bridge_optionC_adaptive(
        n_qubits=n_qubits, qmpo=qmpo, mps_hf=hf_psi,
        wgs_L=wgs_L_p, wgs_R=wgs_R_p,
        A_left=A_left_p, A_right=A_right_p,
        Sg_old_seed=Sg_seed_p, wmpo_H=wmpo,
        rH=256, regN=1e-10, tol=1e-3, maxiter=200, block=6,
        outer_rounds=(1 if USE_ONLY_MPS_SAMPLES else 8), add_per_round=1, max_pairs=50000,
        seed=BRIDGE_SEED, verbose=False,
        dense_threshold=1000, inner_method="lobpcg",
        plot_convergence=False,
    )
    return dict(chi=chi_ref, n_samples=n_smp, sampling_seed=smp_seed,
                n_excitations=len(pool_c), energy=float(best_p["E0"]))


def aggregate_fig5_runs(sweep):
    """Mean/std over sampling seeds for each (chi, n_samples) point."""
    groups = {}
    for r in sweep["runs"]:
        groups.setdefault((r["chi"], r["n_samples"]), []).append(r)
    agg = {"molecule": sweep["molecule"], "seed_grid": list(SEED_GRID), "chi": {}}
    for (chi_v, n_smp), rs in sorted(groups.items()):
        es = np.array([r["energy"] for r in rs], dtype=float)
        nx = np.array([r["n_excitations"] for r in rs], dtype=float)
        blk = agg["chi"].setdefault(str(chi_v), {"E_dmrg": sweep["E_dmrg"][str(chi_v)],
                                                 "points": []})
        blk["points"].append({
            "n_samples": int(n_smp),
            "n_seeds": len(rs),
            "E_mean": float(es.mean()),
            "E_std": float(es.std(ddof=1)) if len(es) > 1 else 0.0,
            "n_exc_mean": float(nx.mean()),
        })
    return agg


SWEEP_FILE = os.path.join(RESULTS_DIR, f"fig5_sweep_{MOLECULE_KEY}.json")
SWEEP_AGG_FILE = os.path.join(RESULTS_DIR, f"fig5_sweep_{MOLECULE_KEY}_agg.json")

def _save_sweep(sweep):
    """Persist raw + aggregated sweep to Drive/results after every point (crash-safe)."""
    with open(SWEEP_FILE, "w") as f:
        json.dump(sweep, f, indent=2)
    with open(SWEEP_AGG_FILE, "w") as f:
        json.dump(aggregate_fig5_runs(sweep), f, indent=2)


if RUN_FIG5_SWEEP:
    # ---- Resume: reload any partial sweep so a disconnect never wastes work ----
    if os.path.exists(SWEEP_FILE):
        with open(SWEEP_FILE) as f:
            sweep = json.load(f)
        print(f"resuming: {len(sweep['runs'])} points already done in {SWEEP_FILE}")
    else:
        sweep = {"molecule": MOLECULE_KEY, "E_dmrg": {}, "runs": []}
    done = {(r["chi"], r["n_samples"], r["sampling_seed"]) for r in sweep["runs"]}

    import time as _time
    total = len(CHI_GRID) * len(N_SAMPLES_GRID) * len(SEED_GRID)
    for chi_ref in CHI_GRID:
        # Skip DMRG entirely if every point for this chi is already done (resume).
        if all((chi_ref, n_smp, smp_seed) in done
               for n_smp in N_SAMPLES_GRID for smp_seed in SEED_GRID):
            print(f"chi={chi_ref}: all points already done, skipping")
            continue
        # DMRG reference for this chi (only if not already recorded)
        if str(chi_ref) not in sweep["E_dmrg"]:
            psi_ref, E_ref = _dmrg_ground_state_from_mpo(wmpo, chi_max=chi_ref)
            sweep["E_dmrg"][str(chi_ref)] = float(E_ref)
            _save_sweep(sweep)
        else:
            psi_ref, _ = _dmrg_ground_state_from_mpo(wmpo, chi_max=chi_ref)
        for n_smp in N_SAMPLES_GRID:
            for smp_seed in SEED_GRID:
                if (chi_ref, n_smp, smp_seed) in done:
                    continue
                t0 = _time.perf_counter()
                rec = run_fig5_point(psi_ref, chi_ref, n_smp, smp_seed)
                rec["wall_s"] = round(_time.perf_counter() - t0, 1)
                sweep["runs"].append(rec)
                done.add((chi_ref, n_smp, smp_seed))
                _save_sweep(sweep)                        # <-- save after EVERY point
                print(f"[{len(sweep['runs']):3d}/{total}] {rec}")
    print("done. saved", SWEEP_FILE, "and", SWEEP_AGG_FILE)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

# ----- Load Fig.-5 data: prefer the multi-seed sweep, fall back to submission-1 -----
# Unified per-chi record: E_dmrg + list of (n_samples, n_exc, E_mean, E_std|None)
if os.path.exists(SWEEP_AGG_FILE):
    with open(SWEEP_AGG_FILE) as f:
        _raw = json.load(f)
    data = {int(c): {"E_dmrg": float(b["E_dmrg"]),
                     "pts": [(p["n_samples"], p["n_exc_mean"], p["E_mean"], p["E_std"])
                             for p in b["points"]]}
            for c, b in _raw["chi"].items()}
    _src_label = "multi-seed sweep (mean \u00b1 1 std over seeds)"
else:
    with open(FIG5_DATA_FILE) as f:
        _raw = json.load(f)
    data = {int(c): {"E_dmrg": float(b["E_dmrg"]),
                     "pts": [(n_smp, n_exc, e, None) for n_exc, n_smp, e in b["points"]]}
            for c, b in _raw["chi"].items()}
    _src_label = "submission-1 data (single run per point)"
print("plotting:", _src_label)

mpl.rcParams.update({
    "font.family":      "serif",
    "font.serif":       ["STIXGeneral", "DejaVu Serif", "Times"],
    "mathtext.fontset": "stix",
    "font.size":        13,
    "axes.labelsize":   15,
    "legend.fontsize":  12,
    "xtick.labelsize":  12,
    "ytick.labelsize":  12,
})

colors = {8: '#2166ac', 32: '#d6604d'}

fig, ax = plt.subplots(figsize=(8, 6))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

chis = sorted(data)
n_samples_ticks = sorted({p[0] for chi in chis for p in data[chi]["pts"]})

# ----- curves with error bars (mean +/- 1 std over seeds, when available) -----
curve_handles = []
for chi in chis:
    pts = sorted(data[chi]["pts"], key=lambda p: p[0])
    x = np.array([p[0] for p in pts], dtype=float)
    y = np.array([p[2] for p in pts], dtype=float)
    s = np.array([np.nan if p[3] is None else p[3] for p in pts], dtype=float)
    yerr = s if (np.isfinite(s).all() and s.max() > 0) else None

    h = ax.errorbar(x, y, yerr=yerr, fmt='s-', markersize=6, linewidth=1.6,
                    color=colors.get(chi, None), ecolor=colors.get(chi, None),
                    elinewidth=1.2, capsize=3, capthick=1.2, zorder=3,
                    label=fr'$\chi={chi}$')
    curve_handles.append(h)

# ----- DMRG reference lines -----
dmrg_handles = []
for chi in chis:
    e = data[chi]["E_dmrg"]
    h, = ax.plot([], [], color=colors.get(chi, None), linestyle='--',
                 linewidth=1.2, alpha=0.9,
                 label=fr'$E_{{\mathrm{{DMRG}}}},\ \chi={chi}$')
    dmrg_handles.append(h)
    ax.axhline(y=e, color=colors.get(chi, None), linestyle='--',
               linewidth=1.2, alpha=0.9, zorder=2)

# ----- pool-size annotations, placed clear of the error bars -----
# Upper curve labels go above its caps, lower curve labels below, so numbers
# never overlap the error bars (or the other curve).
_mean_e = {chi: float(np.mean([p[2] for p in data[chi]["pts"]])) for chi in chis}
_upper_chi = max(_mean_e, key=_mean_e.get)
_lab_margin = 0.0010
for chi in chis:
    _above = (chi == _upper_chi)
    for n_smp, n_exc, e_val, _s in data[chi]["pts"]:
        _err = 0.0 if (_s is None or not np.isfinite(_s)) else float(_s)
        if _above:
            _yt, _va = e_val + _err + _lab_margin, 'bottom'
        else:
            _yt, _va = e_val - _err - _lab_margin, 'top'
        ax.text(n_smp, _yt, str(int(round(n_exc))),
                color=colors.get(chi, None), fontsize=10,
                ha='center', va=_va, rotation=45)

ax.tick_params(axis='y', labelsize=16)
ax.set_xticks(n_samples_ticks)
ax.set_xticklabels(n_samples_ticks, rotation=45, ha='right', fontsize=16)
ax.set_xlabel(r'$\mathcal{N}_{\mathrm{samples}}$', labelpad=8, size=22)
ax.set_ylabel(r'Energy [Ha]', labelpad=8, size=19)

for side in ('top', 'right', 'bottom', 'left'):
    ax.spines[side].set_visible(True)
    ax.spines[side].set_color('black')
    ax.spines[side].set_linewidth(0.9)
ax.tick_params(which='both', colors='black', direction='out',
               top=False, right=False)

ax.legend(handles=curve_handles + dmrg_handles,
          loc='lower left',
          bbox_to_anchor=(0.02, 0.1),
          bbox_transform=ax.transAxes,
          frameon=True, framealpha=0.0,
          fontsize=14, ncol=2,
          columnspacing=1.0, handletextpad=0.5)

ax.grid(True, linestyle=':', alpha=0.5)
plt.subplots_adjust(right=0.80, bottom=0.20)

plt.savefig(os.path.join(FIGS_DIR, f'fig5_excitation_vs_samples_{_raw["molecule"]}.pdf'),
            bbox_inches='tight', facecolor='white')
plt.show()


In [ ]:
# ----- Table of key numbers (referee item 5.2-5) -----
hdr = (f"{'chi':>4s} {'N_samples':>10s} {'N_exc':>6s} "
       f"{'E_mean [Ha]':>18s} {'E_std [Ha]':>12s} {'E-E_dmrg [Ha]':>15s}")
print(hdr)
print("-" * len(hdr))
for chi_v in sorted(data):
    e_ref = data[chi_v]["E_dmrg"]
    for n_smp, n_exc, e_mean, e_std in sorted(data[chi_v]["pts"]):
        std_s = f"{e_std:12.2e}" if e_std is not None else f"{'---':>12s}"
        print(f"{chi_v:4d} {n_smp:10d} {int(round(n_exc)):6d} "
              f"{e_mean:18.10f} {std_s} {e_mean - e_ref:15.6e}")
    print(f"{chi_v:4d} {'DMRG':>10s} {'-':>6s} {e_ref:18.10f}")


## 8. Collect all paper artifacts

Lists every generated figure and data file. On Colab the outputs already persist on Google
Drive (`DRIVE_OUT`); this cell additionally offers a single zip download, whose contents can
be dropped directly into the paper's `resubmission/` folder.


In [ ]:
# ----- Collect generated figures + data for the resubmission -----
import glob
import shutil

artifacts = (sorted(glob.glob(os.path.join(RESULTS_DIR, "*.json")))
             + sorted(glob.glob(os.path.join(FIGS_DIR, "*.pdf")))
             + sorted(glob.glob(os.path.join(FIGS_DIR, "*.png"))))
print(f"{len(artifacts)} artifacts:")
for a in artifacts:
    print("   ", a)

if IN_COLAB:
    # Stage into one folder, zip, and offer a browser download.
    stage = f"/content/mpo_lcu_outputs_{MOLECULE_KEY}"
    shutil.rmtree(stage, ignore_errors=True)
    os.makedirs(f"{stage}/results", exist_ok=True)
    os.makedirs(f"{stage}/figs", exist_ok=True)
    for a in artifacts:
        sub = "results" if a.endswith(".json") else "figs"
        shutil.copy2(a, f"{stage}/{sub}/")
    archive = shutil.make_archive(stage, "zip", root_dir=stage)
    print("\narchive:", archive)
    try:
        from google.colab import files
        files.download(archive)
    except Exception as exc:  # e.g. non-interactive execution
        print("browser download skipped:", exc)


## Appendix A — Operator-side sampling from the Hamiltonian MPO (exploratory)

Complementary to Sec. 4: perfect sampling of Pauli strings from the *Hamiltonian MPO*
(operator support) rather than from the pretrained MPS. Not used for the paper's main figures.


In [ ]:
import numpy as np

# ---------- Pauli basis (d=2) ----------
I = np.array([[1, 0], [0, 1]], dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
ops = np.stack([I, X, Y, Z], axis=0)  # (4,2,2)
PAULI_CHARS = np.array(["I","X","Y","Z"])


# ============================================================
# ONE CELL: Perfect sampling of Pauli strings from an MPO
#   Input MPO site tensors: Ws[i] shape (d,d,a_l,a_r) = (2,2,Dl,Dr)
#   Samples strings with p(σ) ∝ |c_σ|^power, power ∈ {1,2}
#   Returns dict: { "IXYZ...": {"count":..., "coeff_sum":..., "prob_sum":...}, ... }
# ============================================================

def mpo_site_ddLR_to_pauli_channel(W_ddLR: np.ndarray, ops: np.ndarray) -> np.ndarray:
    """
    W_ddLR: (2,2,Dl,Dr) (assumed order: out,in,Dl,Dr)
    ops   : (4,2,2) Paulis [I,X,Y,Z]
    Return Wp: (Dl,4,Dr) where Wp[l,k,r] = 0.5 * Tr(ops[k]^† W[:,:,l,r])
    """
    if W_ddLR.shape[0:2] != (2,2):
        raise ValueError(f"Expected (2,2,Dl,Dr), got {W_ddLR.shape}")
    coeffs_kLR = 0.5 * np.einsum("kab,abLR->kLR", ops.conj(), W_ddLR, optimize=True)
    return np.transpose(coeffs_kLR, (1,0,2))  # (Dl,4,Dr)

def mpo_to_pauli_channel(Ws_ddLR, ops):
    return [mpo_site_ddLR_to_pauli_channel(W, ops) for W in Ws_ddLR]

def _precompute_right_messages(Ws_w):
    """
    Ws_w[i]: (Dl,4,Dr) nonnegative weights (e.g. abs or abs^2)
    R[i]    : (Dl,) backward message; R[L] : (Dr_last,)
    """
    L = len(Ws_w)
    R = [None]*(L+1)
    R[L] = np.ones(Ws_w[-1].shape[2], dtype=float)
    for i in range(L-1, -1, -1):
        Wi = Ws_w[i]  # (Dl,4,Dr)
        R[i] = np.einsum("lkr,r->l", Wi, R[i+1], optimize=True)
    return R

def sample_one_pauli_string_from_pauli_mpo(Ws_pauli, rng, power=1, left_state=0):
    """
    Ws_pauli[i]: (Dl,4,Dr) complex
    Samples σ with p ∝ ∏_i |Ws_pauli[i][aL,k,aR]|^power
    Returns: (sigma_str, digits, coeff_path, prob)
      coeff_path = product of selected local complex entries along the sampled path
      prob       = exact sampling probability for this draw (under chosen power)
    """
    Ws_w = [np.abs(W)**power for W in Ws_pauli]
    R = _precompute_right_messages(Ws_w)

    aL = left_state
    digits = []
    coeff = 1.0 + 0.0j
    prob = 1.0

    for i, (W, Ww) in enumerate(zip(Ws_pauli, Ws_w)):
        un = Ww[aL, :, :] * R[i+1][None, :]  # (4,Dr)
        Z = float(un.sum())
        if not np.isfinite(Z) or Z <= 0:
            raise RuntimeError(f"No continuation at site {i}, aL={aL}")

        flat = (un.reshape(-1) / Z).astype(float)
        idx = rng.choice(flat.size, p=flat)

        k  = idx // un.shape[1]
        aR = idx %  un.shape[1]

        digits.append(int(k))
        prob *= float(flat[idx])
        coeff *= W[aL, k, aR]
        aL = aR

    sigma = "".join(PAULI_CHARS[digits])
    return sigma, digits, coeff, prob

def sample_pauli_pool_from_mpo(Ws_ddLR, n_samples=10_000, seed=0, power=1, left_state=0):
    """
    Ws_ddLR: list of MPO tensors, each (2,2,Dl,Dr)
    Returns dict: sigma -> {"count", "coeff_sum", "prob_sum"}
    """
    Ws_pauli = mpo_to_pauli_channel(Ws_ddLR, ops)
    rng = np.random.default_rng(seed)

    pool = {}
    for _ in range(n_samples):
        sigma, _, coeff, prob = sample_one_pauli_string_from_pauli_mpo(
            Ws_pauli, rng=rng, power=power, left_state=left_state
        )
        if sigma not in pool:
            pool[sigma] = {"count": 0, "coeff_sum": 0.0+0.0j, "prob_sum": 0.0}
        pool[sigma]["count"] += 1
        pool[sigma]["coeff_sum"] += coeff
        pool[sigma]["prob_sum"] += prob

    return pool, Ws_pauli

# ------------------ Example (uncomment to test) ------------------
# # Build a trivial MPO with one term: Z on every site (product Z...Z)
# L = 6
# Ws = []
# for i in range(L):
#     W = np.zeros((2,2,1,1), dtype=complex)
#     W[:,:,0,0] = Z
#     Ws.append(W)
pool, Ws_pauli = sample_pauli_pool_from_mpo(wmpo.Ms, n_samples=1500, power=1)
print("Unique sampled strings:", len(pool))
top = sorted(pool.items(), key=lambda kv: kv[1]["count"], reverse=True)[:5]
for s, info in top:
     print(s, info["count"], info["coeff_sum"])


In [ ]:
import numpy as np

_char2dig = {'I':0,'X':1,'Y':2,'Z':3}

def _digits_from_term(term: str):
    return [_char2dig[c] for c in term]

def coeff_of_pauli_string_from_Ws_pauli(Ws_pauli, term: str, left_state=0, right_state=0):
    """
    Exact coefficient c_term for the Pauli string 'term' given Pauli-channel MPO tensors Ws_pauli[i] (Dl,4,Dr).
    Boundary condition: pick left bond state and right bond state (usually both 0).
    If right_state=None -> sums over the final bond (rarely what you want, but available).
    """
    digits = _digits_from_term(term)
    v = np.zeros(Ws_pauli[0].shape[0], dtype=complex)
    v[left_state] = 1.0

    for i, k in enumerate(digits):
        Wk = Ws_pauli[i][:, k, :]          # (Dl,Dr)
        v = v @ Wk                         # (Dr,)

    if right_state is None:
        return v.sum()
    return v[right_state]

def pool_as_term_coeff(pool, Ws_pauli=None, *, mode="mean", left_state=0, right_state=0):
    """
    pool: dict like {term: {"count":..., "coeff_sum":..., ...}, ...}
    mode:
      - "mean": coeff = coeff_sum / count   (good if Pauli strings are path-unique in your MPO)
      - "exact": coeff = exact contraction from Ws_pauli (needs Ws_pauli)
    returns: {term: coeff}
    """
    out = {}
    if mode == "mean":
        for term, info in pool.items():
            out[term] = info["coeff_sum"].real / max(info["count"], 1)
        return out

    if mode == "exact":
        if Ws_pauli is None:
            raise ValueError("mode='exact' requires Ws_pauli")
        for term in pool.keys():
            out[term] = coeff_of_pauli_string_from_Ws_pauli(
                Ws_pauli, term, left_state=left_state, right_state=right_state
            )
        return out

    raise ValueError("mode must be 'mean' or 'exact'")





# 1) quick (best if your MPO is trie/automaton so each Pauli string is essentially one path)
D_mean = pool_as_term_coeff(pool, mode="mean")

# 2) exact coefficients for the sampled terms (robust even if multiple internal paths contribute)
D_exact = pool_as_term_coeff(pool, Ws_pauli, mode="exact", left_state=0, right_state=0)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def pauli_label(p):
    nxy = sum(c in "XY" for c in p)
    if nxy == 0:
        return "diag"
    if nxy == 2:
        return "single"
    if nxy == 4:
        return "double"
    return "other"

def pauli_vec(p):
    # one-hot per site (appearance)
    m = {"I":[1,0,0,0], "X":[0,1,0,0], "Y":[0,0,1,0], "Z":[0,0,0,1]}
    v = []
    for c in p:
        v += m[c]
    # small summary
    v += [p.count("I"), p.count("X"), p.count("Y"), p.count("Z")]
    return np.array(v, dtype=float)

def _embed_2d(X, random_state=0):
    Xs = StandardScaler().fit_transform(X)
    n, d = Xs.shape

    # tiny groups
    if n == 1:
        return np.array([[0.0, 0.0]])
    if n == 2:
        y1 = PCA(n_components=1).fit_transform(Xs)   # (2,1)
        return np.hstack([y1, np.zeros((2, 1))])     # pad to 2D

    # small groups -> PCA (stable)
    if n < 6:
        k = min(2, n, d)
        Y = PCA(n_components=k).fit_transform(Xs)
        if Y.shape[1] == 1:
            Y = np.hstack([Y, np.zeros((n, 1))])
        return Y

    # larger groups -> t-SNE (local separation)
    perp = min(15, max(3, n // 5))
    return TSNE(
        n_components=2,
        perplexity=perp,
        init="pca",
        learning_rate="auto",
        random_state=random_state
    ).fit_transform(Xs)

def plot_pauli_separate(pauli_dict, max_other=200, random_state=0):
    paulis = list(pauli_dict.keys())
    coeffs = np.array([pauli_dict[p] for p in paulis], dtype=complex)
    mags = np.abs(coeffs)
    labels = [pauli_label(p) for p in paulis]

    groups = ["diag", "single", "double", "other"]
    fig, axes = plt.subplots(2, 2, figsize=(8, 7))
    axes = axes.ravel()

    for ax, g in zip(axes, groups):
        idx = [i for i, lab in enumerate(labels) if lab == g]

        # cap "other" so panel doesn't become too crowded
        if g == "other" and len(idx) > max_other:
            idx = sorted(idx, key=lambda i: mags[i], reverse=True)[:max_other]

        if len(idx) == 0:
            ax.set_title(f"{g} (0)")
            ax.axis("off")
            continue

        Xg = np.vstack([pauli_vec(paulis[i]) for i in idx])
        Yg = _embed_2d(Xg, random_state=random_state)

        mg = mags[idx]
        s = 20 + 120 * (mg / (mg.max() + 1e-12))

        ax.scatter(Yg[:, 0], Yg[:, 1], s=s, alpha=0.8)
        ax.set_title(f"{g} ({len(idx)})")
        ax.set_xticks([])
        ax.set_yticks([])

    fig.suptitle("Pauli appearance maps (separated by excitation class)")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_pauli_separate(D_mean)

## Appendix B — MPO construction benchmark (random Pauli sums)

Timing/bond-dimension benchmark of `get_mpo` on random Hamiltonians. Set the flag to run.


In [ ]:
RUN_MPO_BENCHMARK = False

if RUN_MPO_BENCHMARK:
    N = 20
    n_terms = 400
    rng = np.random.default_rng(1234)

    H = build_random_like_renormalizer(N, n_terms, rng)

    epsilon = 1e-7
    chi_max = 400

    t0 = time.perf_counter()
    wmpo, norm_mpo, discarded = get_mpo(
        N_qubit=N,
        sys_Ham=H,
        epsilon=epsilon,
        chi_max=chi_max,
        show_info=True,   # show your builder's debug info
    )
    t1 = time.perf_counter()

    b = bonds_my(wmpo)
    print("\n=== MY get_mpo ===")
    print("wmpo type:", type(wmpo))
    print("build time [s]:", t1 - t0)
    print("discarded:", discarded)
    print("max bond:", max(b))
    print("bonds:", b)

    # Optional: if your MPO class has a method to apply/evolve, add it here.
    # If not, keep this script focused on MPO construction metrics.
